<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu defterin Türkçesini Colab'da açın](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-04/NB04_transfer_ogrenme.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html)

# Week 04: Transfer learning

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lab.html) · [PDF notes](Week04_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-04/NB04_transfer_learning.ipynb)

</div>

**This notebook has two parts.** In the first, an artificial intelligence (AI) assistant writes the code from a prompt, a written instruction. In the second, you write the code yourself in Python. Both parts reach the same results with the same data, so choose one of them at each practice stop of the lecture page.

| | For whom? | What you do |
|---|---|---|
| **[Part 1: Prompt path](#part-1)** | Participants who do not write code | An AI assistant writes the code from ready prompts; you run it and check the result. |
| **[Part 2: Python path](#part-2)** | Participants who want to learn Python | You write and run the code step by step, with exercises. |

**Where it runs.** The same notebook runs in Google Colab, with nothing to install, and in Jupyter on your own computer (Anaconda; see SETUP.md in the course folder). Choose the one you prefer. In Colab, save your own copy first with *File → Save a copy in Drive*; otherwise the changes are lost when the tab is closed. Fine-tuning a network of four million weights takes about a minute, and training the U-Net about three minutes, on an ordinary laptop; no graphics card is needed. Run Section 0 first, whichever part you choose. It prepares the course files and the packages.

**Data and privacy.** Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution and with the approval of an ethics committee, and never in a cloud notebook or in an AI chat.

## 0. Setup

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub. The second cell loads the packages, chooses the language of the messages and checks the optional packages of the week. The second cell downloads what is missing: the ACDC files of Week 3 (about 105 MB), the apical echocardiograms of 300 patients of CAMUS with their expert outlines (about 227 MB) from the public database of the Human Heart Project (CREATIS, Lyon), and the ImageNet weights of EfficientNet-B0 (about 21 MB). The first run takes a few minutes; later runs skip the files that are already there. ACDC and CAMUS are free for non-commercial scientific research under CC BY-NC-SA 4.0. Every use of CAMUS cites Leclerc et al., IEEE Transactions on Medical Imaging, 2019.

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import math

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic import acdc, camus

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
except ImportError as error:
    raise ImportError("PyTorch or torchvision is not installed. Install them as SETUP.md describes, then restart the kernel.") from error
try:
    import skimage
except ImportError as error:
    raise ImportError("scikit-image is not installed. Install it with pip install scikit-image, then restart the kernel.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 5, 0):
    raise ImportError("The course package vclinic is older than this notebook. Update the course folder (git pull, or download the ZIP file again) and restart the kernel.")
teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
GROUPS = ("NOR", "DCM", "HCM")
acdc.download(GROUPS)
camus.download()
EfficientNet_B0_Weights.IMAGENET1K_V1.get_state_dict(progress=False)
print("ImageNet weights of EfficientNet-B0 are ready.")
print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
for name in ["ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        print(f"{name}: not installed (optional; the notebook uses a built-in fallback)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 4.1": (46657, 0),
    "Exercise 4.2": (28.8, 0.05),
    "Exercise 4.3": (3843, 0),
    "Exercise A": (5288548, 0),
    "Exercise B": (1.0, 0.001),
    "Exercise C": (135, 0),
    "Exercise D": (0.687, 0.001),
    "Exercise E": (8.7, 0.05),
    "Exercise F": (0.789, 0.001),
    "Exercise G": (483170, 0),
    "Exercise H": (79.0, 0),
    "Exercise I": (7.0, 0.05),
    "Exercise J": (159.0, 0),
    "Step 1.1": (0.867, 0.034),
    "Step 1.2": (0.967, 0.1),
    "Step 1.3": (0.62, 0.12),
    "Step 2.1": (0.877, 0.001),
    "Step 2.2": (0.924, 0.05),
    "Step 2.3": (0.908, 0.03),
    "Step 3.1": (42.1, 0.05),
    "Step 3.2": (-4.24, 2.5),
    "Step 3.3": (50.3, 0.05),
})
print("Expected answers are ready.")

In [ ]:
#@title Tables of the exercise prompts (run this cell; the Python path does not need to read it)
def _disc_rows(names):
    """The 20 discs of the four expert outlines of each patient, at the original size of the frames."""
    rows = []
    for name in names:
        for view in ("2CH", "4CH"):
            for phase in ("ED", "ES"):
                _, gt, spacing = camus.load(name, view, phase)
                diameters, thickness, _, _ = camus.discs(gt == 1, spacing)
                for k, d in enumerate(diameters, start=1):
                    rows.append({"patient": name, "view": view, "phase": phase, "disc": k,
                                 "diameter": float(d), "long axis": float(thickness * 20)})
    return pd.DataFrame(rows)


def exercise_table(key):
    """The table "data" of the prompt of an exercise, built from the data of the Python path."""
    g = globals()
    k = key
    if k == "4.1":
        return pd.DataFrame(camus.load("patient0201", "4CH", "ED")[1])
    if k == "4.2":
        return pd.DataFrame(camus.load("patient0201", "4CH", "ES")[1])
    if k == "B":
        names = {"train": "training", "val": "validation", "test": "test"}
        f = g["features"]
        return pd.concat([pd.DataFrame({"patient": g["patients"], "group": [["NOR", "DCM", "HCM"][c] for c in g["y"]],
                                        "set": [names[s] for s in g["sets"]]}),
                          pd.DataFrame(f, columns=[f"f{i}" for i in range(1, f.shape[1] + 1)])], axis=1)
    if k == "D":
        return pd.DataFrame([{"patients per group": n, "draw": d, "accuracy": a}
                             for n, accs in g["curve"].items() for d, a in enumerate(accs, start=1)])
    if k == "E":
        names = {"training": "training", "validation": "validation", "testing": "test"}
        return pd.DataFrame({"patient": g["frame_patient"], "view": g["frame_view"], "phase": g["frame_phase"],
                             "set": [names[s] for s in g["frame_split"]], "outline": list(g["masks"])})
    if k == "F":
        return pd.DataFrame({"end-diastole": [camus.load("patient0201", "4CH", "ED")[1]],
                             "end-systole": [camus.load("patient0201", "4CH", "ES")[1]]})
    if k in ("H", "J"):
        return _disc_rows(["patient0201"]).drop(columns="patient")
    if k == "I":
        if "_discs_test_cache" not in g:
            g["_discs_test_cache"] = _disc_rows(camus.patients("testing"))
        return g["_discs_test_cache"]
    raise KeyError(k)
print("The tables of the exercise prompts are ready.")

<a id="part-1"></a>

# Part 1: Prompt path

**For whom?** This part is for participants who do not write code. Here you type no code. An artificial intelligence (AI) assistant writes it from a ready prompt, a written instruction; you run the code and check the result.

**What you need.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab: Open it with the Gemini button at the top of the window. ChatGPT, Claude, Copilot or Gemini in another browser tab work as well.

**How a step works.**

1. Read the goal of the step.
2. Run the data cell under the prompt. It prepares the table "data" that the prompt describes and shows its first and last rows.
3. Copy the prompt: Select it with the mouse and press Ctrl+C (Cmd+C on a Mac).
4. Paste it into the AI assistant and send it.
5. Copy the code that the assistant returns, paste it into the cell under the data cell in place of the line that starts with `#`, and run the cell with the ▶ button or with Shift+Enter.
6. Run the check cell below it. It prints *correct ✓* or *not yet ✗*.
7. If it prints *not yet*, send the repair prompt below to the assistant together with the code. If the result is still wrong, run the *Ready code* cell. It always gives the correct result, so the next step can start.

**"data" and "answer".** As in Week 1, the data cell of each step prepares the table "data" that the prompt describes. The assistant keeps the result under the name "answer", and the check cell reads it from there.

**At the start of every stop** run its preparation cell once. It loads the data of the stop, so a stop can also be started on another day.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-04/PROMPTS.md), where GitHub shows a copy button next to each one.

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

## Stop 1: Learning from few patients with a network trained on photographs

*Lecture: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#stop-1)* · *The same result on the Python path: Part 2, Sections 1 to 4*

A network trained on more than a million photographs of ImageNet already knows edges, textures and shapes. In this stop EfficientNet-B0 is reused in two ways on the cardiomyopathy groups of ACDC: Its features feed a simple model, and the whole network is fine-tuned. Then only five patients per group are kept, to see how much the result depends on the patients that happen to be chosen.

**At the end of this stop:** The accuracy of features with a logistic regression on the 30 test patients (0.867), the accuracy of the fine-tuned network (about 0.967) and the mean accuracy with five patients per group (0.62). Fine-tuning takes about a minute.

In [ ]:
#@title Preparation of Stop 1: run this cell once (it loads the data of the stop)
GROUPS = ("NOR", "DCM", "HCM")
P = acdc.pairs(GROUPS)
X, y, patients, sets, classes = P["X"], P["y"], P["patient"], P["set"], P["classes"]

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_SD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def as_photos(x, size=128):
    """ED, ES and their mean as three colour channels, resized and normalised like ImageNet."""
    x = torch.as_tensor(x, dtype=torch.float32)
    x3 = torch.stack([x[:, 0], x[:, 1], (x[:, 0] + x[:, 1]) / 2], dim=1)
    x3 = F.interpolate(x3, size=(size, size), mode="bilinear", align_corners=False)
    return (x3 - IMAGENET_MEAN) / IMAGENET_SD

def by_patient(prob, mask):
    """Mean probabilities over the slices of each patient, and the share of correct decisions."""
    names = sorted(set(patients[mask]))
    mean_prob = np.array([prob[patients[mask] == n].mean(axis=0) for n in names])
    truth = np.array([y[mask][patients[mask] == n][0] for n in names])
    return float((mean_prob.argmax(axis=1) == truth).mean())


def _course_features():
    """The 1280 features of EfficientNet-B0 for the 270 slices (computed once)."""
    if "_features_cache" not in globals():
        net = efficientnet_b0(weights=effnet_weights).eval()
        with torch.no_grad():
            globals()["_features_cache"] = torch.cat([net.avgpool(net.features(photos[i:i + 32])).flatten(1)
                                                      for i in range(0, len(photos), 32)]).numpy()
    return _features_cache

photos = as_photos(X)
effnet_weights = EfficientNet_B0_Weights.IMAGENET1K_V1
_SETS = {"train": "training", "val": "validation", "test": "test"}
print(tuple(photos.shape), classes, {s: int((sets == s).sum()) for s in ("train", "val", "test")})
print("Stop 1 is ready.")

### Step 1.1: Features of a network trained on photographs

**Goal.** Describe every slice with the 1280 features of EfficientNet-B0 and learn the three groups from them with a simple model.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: On the cine MR slices of the cardiomyopathy groups of ACDC, the cardiologist wants to know how well the features of a network trained only on photographs separate normal hearts, dilated cardiomyopathy and hypertrophic cardiomyopathy.  
> Data: The notebook holds a table named "data" with 270 short-axis slices of cine MR, the three middle slices of each of the 90 patients of ACDC. Each row is a slice: "patient" gives the patient, "group" the group of the patient ("NOR", "DCM" or "HCM") and "set" the set of the patient ("training", 45 patients, 15 of each group; "validation", 15 patients; "test", 30 patients). Each cell of the column "image" holds the slice prepared for a network trained on photographs: three colour channels (the end-diastolic frame, the end-systolic frame and their mean) of 128 × 128 pixels, normalised with the means and standard deviations of ImageNet.  
> Task: Use EfficientNet-B0 with its ImageNet weights, in evaluation mode. Pass the images of all slices through the network in groups of 32, without recording gradients, and take the output of the average pooling before its last layer: 1,280 numbers per slice, the features. Standardise the features with the training slices and fit a logistic regression on the training slices. Then compute the probabilities of the three groups for the test slices, average them over the three slices of each test patient, and take the group with the largest mean as the decision for the patient.  
> Constraints: Use only this table. For the network, use only the ImageNet weights of EfficientNet-B0 that are already on this computer; no download is needed. The code must not read files, download anything from the internet or install programs.  
> Output: Print the share of the 30 test patients whose decision equals their group with three decimals, and keep it under the name "answer".  
> Test: The features must form a table of 270 rows and 1,280 columns, and there must be 30 test patients.  

In [ ]:
#@title Data of step 1.1: run this cell (it prepares the table "data" and shows it)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": patients, "group": [classes[k] for k in y],
                     "set": [_SETS[s] for s in sets], "image": list(photos.numpy())})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.1
teach.check_step("Step 1.1", "answer", globals())

In [ ]:
#@title Ready code of step 1.1 (run it only if the code of the assistant does not pass the check)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": patients, "group": [classes[k] for k in y],
                     "set": [_SETS[s] for s in sets], "image": list(photos.numpy())})


def accuracy_by_patient(prob, names, truth):
    table = pd.DataFrame(prob)
    table["patient"] = names
    means = table.groupby("patient").mean()
    true_group = pd.Series(truth, index=names).groupby(level=0).first()
    return float((means.to_numpy().argmax(axis=1) == true_group.loc[means.index].to_numpy()).mean())


slice_images = torch.tensor(np.stack(data["image"]))
network = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).eval()
with torch.no_grad():
    slice_features = torch.cat([network.avgpool(network.features(slice_images[i:i + 32])).flatten(1)
                                for i in range(0, len(slice_images), 32)]).numpy()
slice_sets = data["set"].to_numpy()
groups = data["group"].to_numpy()
in_train, in_test = slice_sets == "training", slice_sets == "test"
logistic = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(slice_features[in_train], groups[in_train])
test_prob = logistic.predict_proba(slice_features[in_test])
test_codes = pd.Categorical(groups[in_test], categories=logistic.classes_).codes
test_names = data["patient"].to_numpy()[in_test]
answer = round(accuracy_by_patient(test_prob, test_names, test_codes), 3)
print(slice_features.shape, "test patients:", len(set(test_names)), "accuracy:", answer)
teach.check_step("Step 1.1", "answer", globals())

**What the result shows.** Features learned from photographs alone place 87 percent of the 30 test patients in their group. The network has never seen a heart; its early layers respond to edges and contrasts, and the logistic regression learns which combinations of its 1280 features separate a dilated, a thick and a normal ventricle.

### Step 1.2: Fine-tuning the whole network

**Goal.** Let every layer of EfficientNet-B0 adapt to the slices, starting from the ImageNet weights. The training takes about a minute.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether adapting every layer of a network trained on photographs to the cine MR slices of ACDC separates the three groups better than its features alone.  
> Data: The notebook holds a table named "data" with 270 short-axis slices of cine MR, the three middle slices of each of the 90 patients of ACDC. Each row is a slice: "patient" gives the patient, "group" the group of the patient ("NOR", "DCM" or "HCM") and "set" the set of the patient ("training", 45 patients, 15 of each group; "validation", 15 patients; "test", 30 patients). Each cell of the column "image" holds the slice prepared for a network trained on photographs: three colour channels (the end-diastolic frame, the end-systolic frame and their mean) of 128 × 128 pixels, normalised with the means and standard deviations of ImageNet.  
> Task: Start from EfficientNet-B0 with its ImageNet weights and replace its last layer with dropout at a rate of 0.3 followed by a new linear layer with three outputs, one for each group. Train the whole network on the training slices for 15 epochs, in shuffled groups of 16 slices, with a learning rate of 0.001 and the cross-entropy loss. As data augmentation, turn every group of slices by a random quarter turn. After every epoch, measure the loss on the validation slices without learning from them, and keep the weights of the epoch with the lowest validation loss. Make the random choices the same in every run by using the number 2026 as the starting value. Then decide for each test patient as a clinic would: Average the probabilities of the three groups over the three slices of the patient and take the group with the largest mean.  
> Constraints: Use only this table. For the network, use only the ImageNet weights of EfficientNet-B0 that are already on this computer; no download is needed. The code must not read files, download anything from the internet or install programs.  
> Output: Print the epoch with the lowest validation loss and the share of the 30 test patients whose decision equals their group, with three decimals; keep the share under the name "answer".  
> Test: After the kept weights are loaded, the validation loss must again have its lowest value.  

In [ ]:
#@title Data of step 1.2: run this cell (it prepares the table "data" and shows it)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": patients, "group": [classes[k] for k in y],
                     "set": [_SETS[s] for s in sets], "image": list(photos.numpy())})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.2
teach.check_step("Step 1.2", "answer", globals())

In [ ]:
#@title Ready code of step 1.2 (run it only if the code of the assistant does not pass the check)
_SETS = {"train": "training", "val": "validation", "test": "test"}
data = pd.DataFrame({"patient": patients, "group": [classes[k] for k in y],
                     "set": [_SETS[s] for s in sets], "image": list(photos.numpy())})


def accuracy_by_patient(prob, names, truth):
    table = pd.DataFrame(prob)
    table["patient"] = names
    means = table.groupby("patient").mean()
    true_group = pd.Series(truth, index=names).groupby(level=0).first()
    return float((means.to_numpy().argmax(axis=1) == true_group.loc[means.index].to_numpy()).mean())


slice_images = torch.tensor(np.stack(data["image"]))
codes = torch.tensor(data["group"].map({"NOR": 0, "DCM": 1, "HCM": 2}).to_numpy())
slice_sets = data["set"].to_numpy()
in_train, in_val, in_test = slice_sets == "training", slice_sets == "validation", slice_sets == "test"
torch.manual_seed(2026)
tuned_net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
tuned_net.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))
optimiser = torch.optim.Adam(tuned_net.parameters(), lr=1e-3, weight_decay=1e-4)
order_gen = torch.Generator().manual_seed(2026)
x_tr, y_tr = slice_images[in_train], codes[in_train]
best, best_state = float("inf"), None
for epoch in range(15):
    tuned_net.train()
    order = torch.randperm(len(y_tr), generator=order_gen)
    for start in range(0, len(order), 16):
        j = order[start:start + 16]
        turns = int(torch.randint(0, 4, (1,), generator=order_gen))
        loss = F.cross_entropy(tuned_net(torch.rot90(x_tr[j], turns, dims=(2, 3))), y_tr[j])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    tuned_net.eval()
    with torch.no_grad():
        val_loss = F.cross_entropy(tuned_net(slice_images[in_val]), codes[in_val]).item()
    if val_loss < best:
        best, best_state, best_epoch = val_loss, copy.deepcopy(tuned_net.state_dict()), epoch + 1
tuned_net.load_state_dict(best_state)
tuned_net.eval()
with torch.no_grad():
    slice_prob = F.softmax(tuned_net(slice_images[in_test]), dim=1).numpy()
answer = round(accuracy_by_patient(slice_prob, data["patient"].to_numpy()[in_test], codes.numpy()[in_test]), 3)
print("Best epoch:", best_epoch, "lowest validation loss:", round(best, 4), "accuracy:", answer)
teach.check_step("Step 1.2", "answer", globals())

**What the result shows.** In the run of the course, fine-tuning reached about 0.967, the best result of the week on these 30 patients. The training differs slightly from one computer to another, so the check accepts results within 0.1 of this value. Every weight started from photographs and was adjusted to the slices of 45 patients.

### Step 1.3: Five patients per group

**Goal.** See how much a result from few patients depends on which patients are chosen.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see how much a result from only five patients per group depends on which patients happen to be chosen.  
> Data: The notebook holds a table named "data" with 270 short-axis slices of cine MR, the three middle slices of each of the 90 patients of ACDC. Each row is a slice: "patient" gives the patient, "group" the group of the patient ("NOR", "DCM" or "HCM") and "set" the set of the patient ("training", 45 patients, 15 of each group; "validation", 15 patients; "test", 30 patients). The other 1,280 columns, "f1" to "f1280", hold the features of the slice, the output of the average pooling before the last layer of EfficientNet-B0, which was trained on the photographs of ImageNet.  
> Task: Choose five patients at random from each group of the training set, and repeat this choice five times. Make the random choices the same in every run by using the number 2026 as the starting value. For each choice, standardise the features with the slices of the 15 chosen patients only and fit a logistic regression on these slices. Then compute the accuracy on the 30 test patients, with one decision per patient: the group with the largest mean probability over the three slices of the patient.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the five accuracies, and keep their mean, with three decimals, under the name "answer".  
> Test: Every choice must hold 15 different patients, five of each group.  

In [ ]:
#@title Data of step 1.3: run this cell (it prepares the table "data" and shows it)
_SETS = {"train": "training", "val": "validation", "test": "test"}
_F = _course_features()
data = pd.concat([pd.DataFrame({"patient": patients, "group": [classes[k] for k in y], "set": [_SETS[s] for s in sets]}),
                  pd.DataFrame(_F, columns=[f"f{k}" for k in range(1, _F.shape[1] + 1)])], axis=1)
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.3
teach.check_step("Step 1.3", "answer", globals())

In [ ]:
#@title Ready code of step 1.3 (run it only if the code of the assistant does not pass the check)
_SETS = {"train": "training", "val": "validation", "test": "test"}
_F = _course_features()
data = pd.concat([pd.DataFrame({"patient": patients, "group": [classes[k] for k in y], "set": [_SETS[s] for s in sets]}),
                  pd.DataFrame(_F, columns=[f"f{k}" for k in range(1, _F.shape[1] + 1)])], axis=1)


def accuracy_by_patient(prob, names, truth):
    table = pd.DataFrame(prob)
    table["patient"] = names
    means = table.groupby("patient").mean()
    true_group = pd.Series(truth, index=names).groupby(level=0).first()
    return float((means.to_numpy().argmax(axis=1) == true_group.loc[means.index].to_numpy()).mean())


feats = data[[c for c in data.columns if c.startswith("f")]].to_numpy()
groups = data["group"].to_numpy()
names = data["patient"].to_numpy()
slice_sets = data["set"].to_numpy()
in_train, in_test = slice_sets == "training", slice_sets == "test"
rng = np.random.default_rng(2026)
train_names = sorted(set(names[in_train]))
group_of = {n: groups[names == n][0] for n in train_names}
accuracies = []
for draw in range(5):
    chosen = []
    for g in ("NOR", "DCM", "HCM"):
        pool = [n for n in train_names if group_of[n] == g]
        chosen += sorted(rng.choice(pool, size=5, replace=False).tolist())
    rows = np.isin(names, chosen)
    small = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(feats[rows], groups[rows])
    codes = pd.Categorical(groups[in_test], categories=small.classes_).codes
    accuracies.append(accuracy_by_patient(small.predict_proba(feats[in_test]), names[in_test], codes))
    print("choice", draw + 1, len(set(chosen)), "patients, accuracy", round(accuracies[-1], 3))
answer = round(float(np.mean(accuracies)), 3)
print("Mean accuracy:", answer)
teach.check_step("Step 1.3", "answer", globals())

**What the result shows.** Five choices of five patients per group give a mean of 0.62, and the five accuracies lie far apart. With few patients, the result depends on the patients that happen to be chosen; a fair comparison of methods therefore uses the same choices for every method and reports their spread, as the lecture page does.

**Stop 1 is complete.** Go back to the [lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#stop-1) and continue reading after the stop.

## Stop 2: A U-Net on the echocardiograms of CAMUS

*Lecture: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#stop-2)* · *The same result on the Python path: Part 2, Sections 5 to 7*

A segmentation network gives every pixel a label. In this stop the Dice coefficient is first computed by hand, on an outline shifted by a few pixels. Then a small U-Net learns the cavity of the left ventricle from 800 training frames of CAMUS and is measured on the 200 frames of the 50 test patients, by view and by phase.

**At the end of this stop:** The Dice coefficient of a shifted outline (0.877), the mean Dice coefficient of the U-Net on the test frames (about 0.924) and its weakest pair of view and phase (about 0.908). Training takes about three minutes.

In [ ]:
#@title Preparation of Stop 2: run this cell once (it loads the data of the stop)
if "frames_" not in globals():
    frames_ = camus.frames(camus.patients(), size=96)
images, masks = frames_["images"], frames_["masks"]
frame_patient, frame_view, frame_phase, frame_split = frames_["patient"], frames_["view"], frames_["phase"], frames_["split"]
frame_shape = frames_["shape"]

def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Three levels down, a bottleneck and three levels up with skip connections."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Dice coefficient of each frame: two boolean arrays of the same shape."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


def _course_unet():
    """The U-Net of the course, trained once (about three minutes): the probability of the cavity in every pixel of the 200 test frames."""
    if "_unet_cache" in globals():
        return _unet_cache
    cavity = masks == 1
    tr, va, te = frame_split == "training", frame_split == "validation", frame_split == "testing"
    X_img = torch.tensor(images)[:, None]
    Y_cav = torch.tensor(cavity.astype(np.int64))
    torch.manual_seed(RANDOM_STATE)
    unet = UNet()
    epochs, batch = 15, 16
    optimizer = torch.optim.Adam(unet.parameters(), lr=3e-3)
    steps = epochs * ((int(tr.sum()) + batch - 1) // batch)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=steps)
    order_gen = torch.Generator().manual_seed(RANDOM_STATE)
    X_tr, Y_tr = X_img[tr], Y_cav[tr]
    best, best_state = -1.0, None
    for epoch in range(epochs):
        unet.train()
        order = torch.randperm(len(X_tr), generator=order_gen)
        for start in range(0, len(X_tr), batch):
            j = order[start:start + batch]
            out = unet(X_tr[j])
            loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            schedule.step()
        unet.eval()
        with torch.no_grad():
            val_dice = float(dice(unet(X_img[va]).argmax(dim=1).numpy() == 1, cavity[va]).mean())
        if val_dice > best:
            best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
    unet.load_state_dict(best_state)
    unet.eval()
    with torch.no_grad():
        test_prob = F.softmax(unet(X_img[te]), dim=1)[:, 1].numpy()
    globals()["_unet_cache"] = test_prob
    return test_prob


def _course_agreement():
    """Volumes and EF of the 50 test patients from the outlines of the expert and of the U-Net of the course (computed once)."""
    if "_agreement_cache" in globals():
        return _agreement_cache
    prob = _course_unet()
    outlines = {}
    for k, i in enumerate(np.where(frame_split == "testing")[0]):
        outlines.setdefault(frame_patient[i], {})[(frame_view[i], frame_phase[i])] = camus.to_original(prob[k], frame_shape[i])
    rows = []
    for name in camus.patients("testing"):
        ref, net = camus.measure(name), camus.measure(name, outlines[name])
        rows.append({"patient": name, "edv_expert": ref["edv_ml"], "edv_net": net["edv_ml"], "esv_expert": ref["esv_ml"],
                     "esv_net": net["esv_ml"], "ef_expert": ref["ef"], "ef_net": net["ef"]})
    globals()["_agreement_cache"] = pd.DataFrame(rows)
    return _agreement_cache
_FSETS = {"training": "training", "validation": "validation", "testing": "test"}

print(images.shape, {s: int((frame_split == s).sum()) for s in ("training", "validation", "testing")})
print("Stop 2 is ready.")

### Step 2.1: The Dice coefficient by hand

**Goal.** See how much a small shift of an outline lowers the Dice coefficient.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see how much a small shift of an outline lowers the Dice coefficient, the usual measure of agreement between two outlines.  
> Data: The notebook holds a table named "data" with 96 rows and 96 columns: the expert outline of the cavity of the left ventricle in the apical four-chamber frame at end-diastole of patient 201 of CAMUS, resized to 96 × 96 pixels. Each cell is a pixel: 1 inside the cavity and 0 outside.  
> Task: Shift the cavity by 3 pixels to the right, that is, move every pixel 3 columns to the right. Count the pixels of the original and of the shifted cavity and the pixels that the two share. Compute the Dice coefficient: twice the shared pixels divided by the sum of the two counts.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the three counts, and keep the Dice coefficient, with three decimals, under the name "answer".  
> Test: The Dice coefficient of the cavity with itself, without a shift, must be 1.  

In [ ]:
#@title Data of step 2.1: run this cell (it prepares the table "data" and shows it)
_i = int(np.where((frame_patient == "patient0201") & (frame_view == "4CH") & (frame_phase == "ED"))[0][0])
data = pd.DataFrame((masks[_i] == 1).astype(int))
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.1
teach.check_step("Step 2.1", "answer", globals())

In [ ]:
#@title Ready code of step 2.1 (run it only if the code of the assistant does not pass the check)
_i = int(np.where((frame_patient == "patient0201") & (frame_view == "4CH") & (frame_phase == "ED"))[0][0])
data = pd.DataFrame((masks[_i] == 1).astype(int))
a = data.to_numpy() == 1
b = np.roll(a, 3, axis=1)
shared = int((a & b).sum())
answer = round(2 * shared / (int(a.sum()) + int(b.sum())), 3)
print("pixels:", int(a.sum()), int(b.sum()), "shared:", shared, "Dice:", answer,
      "without a shift:", 2 * int(a.sum()) / (2 * int(a.sum())))
teach.check_step("Step 2.1", "answer", globals())

**What the result shows.** A shift of 3 pixels, about 3 mm on a frame of 96 pixels, lowers the Dice coefficient to 0.877. Small cavities lose more: The same shift costs a larger share of the pixels of an end-systolic cavity, which is one reason for lower Dice coefficients at ES.

### Step 2.2: Training the U-Net

**Goal.** Train a small U-Net to draw the cavity of the left ventricle and measure it on frames of patients it has never seen. The training takes about three minutes.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to train a small U-Net that outlines the cavity of the left ventricle on the apical echocardiograms of CAMUS, and to measure it on patients that the network has never seen.  
> Data: The notebook holds a table named "data" with 1,200 frames of CAMUS: the end-diastolic and end-systolic frames of the apical two-chamber and four-chamber views of 300 patients. Each row is a frame: "patient" gives the patient, "view" the view ("2CH" or "4CH"), "phase" the phase ("ED" or "ES") and "set" the set of the patient ("training", 200 patients; "validation", 50 patients; "test", 50 patients). Each cell of the column "frame" holds the frame as an image of 96 × 96 pixels with values from 0 to 1, and each cell of the column "cavity" the expert outline of the cavity of the left ventricle at the same size: 1 inside the cavity and 0 outside.  
> Task: Build a small U-Net that gives two scores to every pixel, one for the cavity and one for the rest. It has three levels down, a bottleneck and three levels up with skip connections. Every level has two convolutions with 3 × 3 kernels that keep the size of the image, each followed by batch normalisation and ReLU. On the way down, a 2 × 2 max pooling halves the image; on the way up, a 2 × 2 transposed convolution doubles it, and the result is joined with the output of the same level on the way down. The first level has 8 channels, and the number of channels doubles at every level down, up to 64 in the bottleneck. A last 1 × 1 convolution gives the two scores. Train it on the training frames for 15 epochs, in shuffled groups of 16 frames. The loss is the sum of the cross-entropy and the soft Dice loss, which is one minus the Dice coefficient computed from the probabilities of the cavity. Use a one-cycle learning rate with a highest value of 0.003. After every epoch, compute the mean Dice coefficient of the validation frames, and keep the weights of the epoch where it is highest. Make the random choices the same in every run by using the number 2026 as the starting value. Then compute the probability of the cavity in every pixel of the test frames, threshold it at 0.5 and compute the Dice coefficient of each test frame against its expert outline.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of trainable parameters, and keep the mean Dice coefficient of the 200 test frames, with three decimals, under the name "answer".  
> Test: There must be 800 training frames and 200 test frames, and the network must have 121,394 trainable parameters.  

In [ ]:
#@title Data of step 2.2: run this cell (it prepares the table "data" and shows it)
_FSETS = {"training": "training", "validation": "validation", "testing": "test"}
data = pd.DataFrame({"patient": frame_patient, "view": frame_view, "phase": frame_phase,
                     "set": [_FSETS[s] for s in frame_split], "frame": list(images),
                     "cavity": list((masks == 1).astype(np.uint8))})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.2
teach.check_step("Step 2.2", "answer", globals())

In [ ]:
#@title Ready code of step 2.2 (run it only if the code of the assistant does not pass the check)
_FSETS = {"training": "training", "validation": "validation", "testing": "test"}
data = pd.DataFrame({"patient": frame_patient, "view": frame_view, "phase": frame_phase,
                     "set": [_FSETS[s] for s in frame_split], "frame": list(images),
                     "cavity": list((masks == 1).astype(np.uint8))})


def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Three levels down, a bottleneck and three levels up with skip connections."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Dice coefficient of each frame: two boolean arrays of the same shape."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


frame_images = np.stack(data["frame"]).astype(np.float32)
cavities = np.stack(data["cavity"]).astype(np.int64)
frame_sets = data["set"].to_numpy()
in_train, in_val, in_test = frame_sets == "training", frame_sets == "validation", frame_sets == "test"
X_img, Y_cav = torch.tensor(frame_images)[:, None], torch.tensor(cavities)
torch.manual_seed(2026)
unet = UNet()
epochs, batch = 15, 16
optimiser = torch.optim.Adam(unet.parameters(), lr=3e-3)
schedule = torch.optim.lr_scheduler.OneCycleLR(optimiser, max_lr=3e-3, total_steps=epochs * ((int(in_train.sum()) + batch - 1) // batch))
order_gen = torch.Generator().manual_seed(2026)
X_tr, Y_tr = X_img[in_train], Y_cav[in_train]
best, best_state = -1.0, None
for epoch in range(epochs):
    unet.train()
    order = torch.randperm(len(X_tr), generator=order_gen)
    for start in range(0, len(X_tr), batch):
        j = order[start:start + batch]
        out = unet(X_tr[j])
        loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        schedule.step()
    unet.eval()
    with torch.no_grad():
        val_dice = float(dice(unet(X_img[in_val]).argmax(dim=1).numpy() == 1, cavities[in_val] == 1).mean())
    if val_dice > best:
        best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
unet.load_state_dict(best_state)
unet.eval()
with torch.no_grad():
    test_prob = F.softmax(unet(X_img[in_test]), dim=1)[:, 1].numpy()
answer = round(float(dice(test_prob > 0.5, cavities[in_test] == 1).mean()), 3)
print(int(in_train.sum()), int(in_test.sum()), sum(p.numel() for p in unet.parameters() if p.requires_grad),
      "best epoch:", best_epoch, "test Dice:", answer)
teach.check_step("Step 2.2", "answer", globals())

**What the result shows.** In the run of the course the U-Net reached a mean Dice coefficient of about 0.924 on the test frames, with 121,394 weights and 800 training frames. The check accepts results within 0.05 of this value, because training differs slightly from one computer to another.

### Step 2.3: The Dice coefficient by view and phase

**Goal.** Find where the network draws worst, as an echocardiography laboratory would ask. The data cell trains the U-Net of the course once, which takes about three minutes.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: As an echocardiography laboratory would ask, the cardiologist wants to know in which view and phase a U-Net outlines the cavity of the left ventricle worst.  
> Data: The notebook holds a table named "data" with the 200 test frames of CAMUS, the apical two-chamber and four-chamber views of 50 patients at end-diastole and end-systole. Each row is a frame: "patient" gives the patient, "view" the view ("2CH" or "4CH") and "phase" the phase ("ED" or "ES"). Each cell of the column "probability" holds, for every pixel of the frame at 96 × 96 pixels, the probability of the cavity of the left ventricle given by a U-Net, from 0 to 1. Each cell of the column "cavity" holds the expert outline of the cavity at the same size: 1 inside the cavity and 0 outside.  
> Task: Threshold the probability of every frame at 0.5 and compute the Dice coefficient of the frame against its expert cavity: twice the shared pixels divided by the sum of the pixels of the two. Group the frames by view and phase, and print a small table with the number of frames and the mean Dice coefficient of the four groups: 2CH ED, 2CH ES, 4CH ED and 4CH ES.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Keep the lowest of the four means, with three decimals, under the name "answer".  
> Test: Each of the four groups must hold 50 frames, and the mean of the four means must equal the mean Dice coefficient of all 200 frames.  

In [ ]:
#@title Data of step 2.3: run this cell (it prepares the table "data" and shows it)
_p, _te = _course_unet(), frame_split == "testing"
data = pd.DataFrame({"patient": frame_patient[_te], "view": frame_view[_te], "phase": frame_phase[_te],
                     "probability": list(_p), "cavity": list((masks[_te] == 1).astype(np.uint8))})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.3
teach.check_step("Step 2.3", "answer", globals())

In [ ]:
#@title Ready code of step 2.3 (run it only if the code of the assistant does not pass the check)
_p, _te = _course_unet(), frame_split == "testing"
data = pd.DataFrame({"patient": frame_patient[_te], "view": frame_view[_te], "phase": frame_phase[_te],
                     "probability": list(_p), "cavity": list((masks[_te] == 1).astype(np.uint8))})
per_frame = []
for prob, cavity in zip(data["probability"], data["cavity"]):
    pred, true = prob > 0.5, cavity == 1
    per_frame.append(2 * (pred & true).sum() / max(pred.sum() + true.sum(), 1))
data["Dice"] = per_frame
by_group = data.groupby(["view", "phase"])["Dice"].agg(["count", "mean"]).round(3)
print(by_group)
answer = round(float(by_group["mean"].min()), 3)
print("Lowest mean:", answer)
teach.check_step("Step 2.3", "answer", globals())

**What the result shows.** The weakest group has a mean Dice coefficient of about 0.908; in the run of the course it was the two-chamber view at end-systole. At ES the cavity is small, so a border of the same thickness is a larger share of it, and the apex and the lateral wall are often harder to see.

**Stop 2 is complete.** Go back to the [lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#stop-2) and continue reading after the stop.

## Stop 3: The ejection fraction by the method of discs

*Lecture: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#stop-3)* · *The same result on the Python path: Part 2, Sections 8 to 9*

The outlines become numbers with the biplane method of discs, as in an echocardiography laboratory. In this stop the volumes and the ejection fraction (EF) of patient 201 are computed from the expert outlines, the network is compared with the expert on the 50 test patients with a Bland-Altman analysis, and the case is measured again with another convention for the thickness of the discs.

**At the end of this stop:** The EF of patient 201 from the expert outlines (percent 42.1), the mean difference between the network and the expert on 50 patients (about −4.24 percentage points) and the EF of the case with a thickness for each phase (percent 50.3).

In [ ]:
#@title Preparation of Stop 3: run this cell once (it loads the data of the stop)
if "frames_" not in globals():
    frames_ = camus.frames(camus.patients(), size=96)
images, masks = frames_["images"], frames_["masks"]
frame_patient, frame_view, frame_phase, frame_split = frames_["patient"], frames_["view"], frames_["phase"], frames_["split"]
frame_shape = frames_["shape"]

def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Three levels down, a bottleneck and three levels up with skip connections."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Dice coefficient of each frame: two boolean arrays of the same shape."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


def _course_unet():
    """The U-Net of the course, trained once (about three minutes): the probability of the cavity in every pixel of the 200 test frames."""
    if "_unet_cache" in globals():
        return _unet_cache
    cavity = masks == 1
    tr, va, te = frame_split == "training", frame_split == "validation", frame_split == "testing"
    X_img = torch.tensor(images)[:, None]
    Y_cav = torch.tensor(cavity.astype(np.int64))
    torch.manual_seed(RANDOM_STATE)
    unet = UNet()
    epochs, batch = 15, 16
    optimizer = torch.optim.Adam(unet.parameters(), lr=3e-3)
    steps = epochs * ((int(tr.sum()) + batch - 1) // batch)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=steps)
    order_gen = torch.Generator().manual_seed(RANDOM_STATE)
    X_tr, Y_tr = X_img[tr], Y_cav[tr]
    best, best_state = -1.0, None
    for epoch in range(epochs):
        unet.train()
        order = torch.randperm(len(X_tr), generator=order_gen)
        for start in range(0, len(X_tr), batch):
            j = order[start:start + batch]
            out = unet(X_tr[j])
            loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            schedule.step()
        unet.eval()
        with torch.no_grad():
            val_dice = float(dice(unet(X_img[va]).argmax(dim=1).numpy() == 1, cavity[va]).mean())
        if val_dice > best:
            best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
    unet.load_state_dict(best_state)
    unet.eval()
    with torch.no_grad():
        test_prob = F.softmax(unet(X_img[te]), dim=1)[:, 1].numpy()
    globals()["_unet_cache"] = test_prob
    return test_prob


def _course_agreement():
    """Volumes and EF of the 50 test patients from the outlines of the expert and of the U-Net of the course (computed once)."""
    if "_agreement_cache" in globals():
        return _agreement_cache
    prob = _course_unet()
    outlines = {}
    for k, i in enumerate(np.where(frame_split == "testing")[0]):
        outlines.setdefault(frame_patient[i], {})[(frame_view[i], frame_phase[i])] = camus.to_original(prob[k], frame_shape[i])
    rows = []
    for name in camus.patients("testing"):
        ref, net = camus.measure(name), camus.measure(name, outlines[name])
        rows.append({"patient": name, "edv_expert": ref["edv_ml"], "edv_net": net["edv_ml"], "esv_expert": ref["esv_ml"],
                     "esv_net": net["esv_ml"], "ef_expert": ref["ef"], "ef_net": net["ef"]})
    globals()["_agreement_cache"] = pd.DataFrame(rows)
    return _agreement_cache


def _disc_rows(names):
    """The 20 discs of the four expert outlines of each patient, at the original size of the frames."""
    rows = []
    for name in names:
        for view in ("2CH", "4CH"):
            for phase in ("ED", "ES"):
                _, gt, spacing = camus.load(name, view, phase)
                diameters, thickness, _, _ = camus.discs(gt == 1, spacing)
                for k, d in enumerate(diameters, start=1):
                    rows.append({"patient": name, "view": view, "phase": phase, "disc": k,
                                 "diameter": float(d), "long axis": float(thickness * 20)})
    return pd.DataFrame(rows)

print(len(camus.patients("testing")), "test patients")
print("Stop 3 is ready.")

### Step 3.1: Patient 201 by the method of discs

**Goal.** Measure the case of the week from the outlines of the expert, as the reference of CAMUS does.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compute the end-diastolic and end-systolic volumes and the ejection fraction of patient 201 of CAMUS from the expert outlines by the biplane method of discs, as in the reference of CAMUS.  
> Data: The notebook holds a table named "data" with the discs of the method of discs on the four expert outlines of the cavity of the left ventricle of patient 201 of CAMUS: the apical two-chamber (2CH) and four-chamber (4CH) views at end-diastole (ED) and end-systole (ES). Each outline is divided into 20 discs perpendicular to its long axis, from the mitral annulus to the apex. Each row is a disc: "view" and "phase" name the outline, "disc" gives its number from 1 to 20, "diameter" its diameter in mm and "long axis" the length of the long axis of the outline in mm, the same for its 20 discs. The diameters and the lengths were measured at the original size of the frames, with the real size of a pixel.  
> Task: Compute each volume by the biplane method of discs, V = π/4 × Σ (a_i × b_i) × h, where a_i is the diameter of disc i in the 2CH outline, b_i that of the same disc in the 4CH outline of the same phase, and h the thickness of a disc. As in the reference of CAMUS, use one thickness for both phases, the longest of the four long axes divided by 20. Divide by 1,000 to turn mm³ into mL, round each volume to a whole number of mL, and compute the EF from the rounded volumes as (EDV − ESV)/EDV × 100.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print EDV and ESV in mL, and keep the EF, in percent with one decimal, under the name "answer".  
> Test: EDV must be larger than ESV and lie between 50 and 300 mL.  

In [ ]:
#@title Data of step 3.1: run this cell (it prepares the table "data" and shows it)
data = _disc_rows(["patient0201"]).drop(columns="patient")
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.1
teach.check_step("Step 3.1", "answer", globals())

In [ ]:
#@title Ready code of step 3.1 (run it only if the code of the assistant does not pass the check)
data = _disc_rows(["patient0201"]).drop(columns="patient")


def volume(phase_name, thickness):
    part = data[data["phase"] == phase_name].sort_values("disc")
    a = part[part["view"] == "2CH"]["diameter"].to_numpy()
    b = part[part["view"] == "4CH"]["diameter"].to_numpy()
    return round(np.pi / 4 * float(np.sum(a * b)) * thickness / 1000)


thickness = data["long axis"].max() / 20
edv, esv = volume("ED", thickness), volume("ES", thickness)
answer = round(100 * (edv - esv) / edv, 1)
print("EDV:", edv, "mL · ESV:", esv, "mL · EF:", answer, "%")
teach.check_step("Step 3.1", "answer", globals())

**What the result shows.** The expert outlines give an end-diastolic volume of 159 mL, an end-systolic volume of 92 mL and an EF of 42.1 percent, a mildly reduced ejection fraction. The case of the week asks whether the network reaches it as well.

### Step 3.2: Agreement of the network with the expert

**Goal.** Compare the automatic EF with the expert's on the 50 test patients, as a laboratory would before using it. If the U-Net of the course is not yet in memory, the data cell trains it first, which takes about three minutes.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Before an automatic ejection fraction is used, the cardiologist wants to compare it with the expert on the 50 test patients of CAMUS with a Bland-Altman analysis, as a laboratory would.  
> Data: The notebook holds a table named "data" with the 50 test patients of CAMUS. Each row is a patient: "patient" gives the patient, "expert EF" the ejection fraction from the outlines of the expert and "network EF" the ejection fraction from the outlines drawn by a U-Net, both in percent and both by the biplane method of discs.  
> Task: Compute the difference of the EF for every patient, network minus expert, then the mean and the standard deviation of the differences and the limits of agreement, the mean ± 1.96 standard deviations. Draw a Bland-Altman plot: the difference against the mean of the two EFs, with horizontal lines at the mean difference and at the two limits.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the standard deviation and the two limits of agreement, and keep the mean difference, in percentage points with two decimals, under the name "answer".  
> Test: There must be 50 patients, and the expert EF of patient0201 must be 42.1 percent.  

In [ ]:
#@title Data of step 3.2: run this cell (it prepares the table "data" and shows it)
_a = _course_agreement()
data = pd.DataFrame({"patient": _a["patient"], "expert EF": _a["ef_expert"], "network EF": _a["ef_net"]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.2
teach.check_step("Step 3.2", "answer", globals())

In [ ]:
#@title Ready code of step 3.2 (run it only if the code of the assistant does not pass the check)
_a = _course_agreement()
data = pd.DataFrame({"patient": _a["patient"], "expert EF": _a["ef_expert"], "network EF": _a["ef_net"]})
diff = data["network EF"] - data["expert EF"]
mean_ef = (data["network EF"] + data["expert EF"]) / 2
bias, sd = float(diff.mean()), float(diff.std(ddof=1))
low, high = bias - 1.96 * sd, bias + 1.96 * sd
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(mean_ef, diff, s=18, color="#1F5F8B")
for value, style in ((bias, "-"), (low, "--"), (high, "--")):
    ax.axhline(value, color="#9B1B1B", ls=style, lw=1)
ax.set_xlabel("Mean of the two EFs (%)")
ax.set_ylabel("Network minus expert (points)")
plt.show()
answer = round(bias, 2)
print(len(data), "patients; expert EF of patient0201:",
      float(data.set_index("patient").loc["patient0201", "expert EF"]))
print("Mean difference:", answer, "SD:", round(sd, 2), "limits:", round(low, 1), round(high, 1))
teach.check_step("Step 3.2", "answer", globals())

**What the result shows.** In the run of the course the mean difference between the network and the expert was about −4.24 points: On average, the network measured the EF lower than the expert, and the limits of agreement lay about 16 points on either side of the mean difference. A laboratory decides before the test which width it accepts; a correlation alone could not tell whether the two methods can replace each other.

### Step 3.3: Another convention for the thickness of the discs

**Goal.** See how much a convention of the software, not a border, changes the EF of the case.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see how much the convention for the thickness of the discs, not a border, changes the ejection fraction of patient 201.  
> Data: The notebook holds a table named "data" with the discs of the method of discs on the four expert outlines of the cavity of the left ventricle of patient 201 of CAMUS: the apical two-chamber (2CH) and four-chamber (4CH) views at end-diastole (ED) and end-systole (ES). Each outline is divided into 20 discs perpendicular to its long axis, from the mitral annulus to the apex. Each row is a disc: "view" and "phase" name the outline, "disc" gives its number from 1 to 20, "diameter" its diameter in mm and "long axis" the length of the long axis of the outline in mm, the same for its 20 discs. The diameters and the lengths were measured at the original size of the frames, with the real size of a pixel.  
> Task: Compute each volume by the biplane method of discs, V = π/4 × Σ (a_i × b_i) × h, where a_i is the diameter of disc i in the 2CH outline, b_i that of the same disc in the 4CH outline of the same phase, and h the thickness of a disc. This time, use a thickness for each phase, the longer of the two long axes of that phase divided by 20. Divide by 1,000 to turn mm³ into mL, round each volume to a whole number of mL, and compute the EF from the rounded volumes as (EDV − ESV)/EDV × 100. For comparison, also compute EDV with one thickness for both phases, from the longest of the four long axes.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print EDV and ESV in mL, and keep the EF, in percent with one decimal, under the name "answer".  
> Test: The two EDVs must be equal, because at end-diastole both conventions use the same long axis.  

In [ ]:
#@title Data of step 3.3: run this cell (it prepares the table "data" and shows it)
data = _disc_rows(["patient0201"]).drop(columns="patient")
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.3
teach.check_step("Step 3.3", "answer", globals())

In [ ]:
#@title Ready code of step 3.3 (run it only if the code of the assistant does not pass the check)
data = _disc_rows(["patient0201"]).drop(columns="patient")


def volume(phase_name, thickness):
    part = data[data["phase"] == phase_name].sort_values("disc")
    a = part[part["view"] == "2CH"]["diameter"].to_numpy()
    b = part[part["view"] == "4CH"]["diameter"].to_numpy()
    return round(np.pi / 4 * float(np.sum(a * b)) * thickness / 1000)


axes = data.groupby("phase")["long axis"].max()
edv, esv = volume("ED", axes["ED"] / 20), volume("ES", axes["ES"] / 20)
answer = round(100 * (edv - esv) / edv, 1)
print("EDV:", edv, "mL · ESV:", esv, "mL · EF:", answer, "% · EDV with one thickness:",
      volume("ED", data["long axis"].max() / 20), "mL")
teach.check_step("Step 3.3", "answer", globals())

**What the result shows.** With a thickness for each phase, the end-systolic volume falls from 92 to 79 mL and the EF rises to 50.3 percent, although not a single pixel of the outlines changed. Before an automatic EF is compared with a laboratory, the convention of its software must be known.

**Stop 3 is complete.** Go back to the [lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#stop-3) and continue reading after the stop.

## Weekly project on the prompt path

*Project of the week: [README of Week 4, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-04/README.md#weekly-project)*

The weekly project can also be built with the prompts below, one for each step. Run the preparation cell once: It loads the data of the project. Each step then works like a step of the stops, with one difference: The project has no check cell and no ready code, because its results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

In [ ]:
#@title Preparation of the project: run this cell once (it loads the data of the project)
if "frames_" not in globals():
    frames_ = camus.frames(camus.patients(), size=96)
images, masks = frames_["images"], frames_["masks"]
frame_patient, frame_view, frame_phase, frame_split = frames_["patient"], frames_["view"], frames_["phase"], frames_["split"]
frame_shape = frames_["shape"]

def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Three levels down, a bottleneck and three levels up with skip connections."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Dice coefficient of each frame: two boolean arrays of the same shape."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


def _course_unet():
    """The U-Net of the course, trained once (about three minutes): the probability of the cavity in every pixel of the 200 test frames."""
    if "_unet_cache" in globals():
        return _unet_cache
    cavity = masks == 1
    tr, va, te = frame_split == "training", frame_split == "validation", frame_split == "testing"
    X_img = torch.tensor(images)[:, None]
    Y_cav = torch.tensor(cavity.astype(np.int64))
    torch.manual_seed(RANDOM_STATE)
    unet = UNet()
    epochs, batch = 15, 16
    optimizer = torch.optim.Adam(unet.parameters(), lr=3e-3)
    steps = epochs * ((int(tr.sum()) + batch - 1) // batch)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=steps)
    order_gen = torch.Generator().manual_seed(RANDOM_STATE)
    X_tr, Y_tr = X_img[tr], Y_cav[tr]
    best, best_state = -1.0, None
    for epoch in range(epochs):
        unet.train()
        order = torch.randperm(len(X_tr), generator=order_gen)
        for start in range(0, len(X_tr), batch):
            j = order[start:start + batch]
            out = unet(X_tr[j])
            loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            schedule.step()
        unet.eval()
        with torch.no_grad():
            val_dice = float(dice(unet(X_img[va]).argmax(dim=1).numpy() == 1, cavity[va]).mean())
        if val_dice > best:
            best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
    unet.load_state_dict(best_state)
    unet.eval()
    with torch.no_grad():
        test_prob = F.softmax(unet(X_img[te]), dim=1)[:, 1].numpy()
    globals()["_unet_cache"] = test_prob
    return test_prob


def _course_agreement():
    """Volumes and EF of the 50 test patients from the outlines of the expert and of the U-Net of the course (computed once)."""
    if "_agreement_cache" in globals():
        return _agreement_cache
    prob = _course_unet()
    outlines = {}
    for k, i in enumerate(np.where(frame_split == "testing")[0]):
        outlines.setdefault(frame_patient[i], {})[(frame_view[i], frame_phase[i])] = camus.to_original(prob[k], frame_shape[i])
    rows = []
    for name in camus.patients("testing"):
        ref, net = camus.measure(name), camus.measure(name, outlines[name])
        rows.append({"patient": name, "edv_expert": ref["edv_ml"], "edv_net": net["edv_ml"], "esv_expert": ref["esv_ml"],
                     "esv_net": net["esv_ml"], "ef_expert": ref["ef"], "ef_net": net["ef"]})
    globals()["_agreement_cache"] = pd.DataFrame(rows)
    return _agreement_cache
_FSETS = {"training": "training", "validation": "validation", "testing": "test"}

_QUALITY = {"Good": "good", "Medium": "medium", "Poor": "poor"}
_quality = {(r["patient"], v): _QUALITY.get(r["quality_" + v.lower()], r["quality_" + v.lower()])
            for _, r in camus.table().iterrows() for v in ("2CH", "4CH")}
print("The project is ready.")

### Project step 1: Train the U-Net and measure its outlines

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to train a U-Net that outlines the cavity of the left ventricle on the echocardiograms of CAMUS, and to see where it draws worse: by view, by phase and by image quality.  
> Data: The notebook holds a table named "data" with 1,200 frames of CAMUS: the end-diastolic and end-systolic frames of the apical two-chamber and four-chamber views of 300 patients. Each row is a frame: "patient" gives the patient, "view" the view ("2CH" or "4CH"), "phase" the phase ("ED" or "ES"), "set" the set of the patient ("training", 200 patients; "validation", 50 patients; "test", 50 patients) and "image quality" the quality of the view of the patient that the authors gave ("good", "medium" or "poor"). Each cell of the column "frame" holds the frame as an image of 96 × 96 pixels with values from 0 to 1, and each cell of the column "cavity" the expert outline of the cavity of the left ventricle at the same size: 1 inside the cavity and 0 outside.  
> Task: Build a small U-Net that gives two scores to every pixel, one for the cavity and one for the rest. It has three levels down, a bottleneck and three levels up with skip connections. Every level has two convolutions with 3 × 3 kernels that keep the size of the image, each followed by batch normalisation and ReLU. On the way down, a 2 × 2 max pooling halves the image; on the way up, a 2 × 2 transposed convolution doubles it, and the result is joined with the output of the same level on the way down. The first level has 8 channels, and the number of channels doubles at every level down, up to 64 in the bottleneck. A last 1 × 1 convolution gives the two scores. Train it on the training frames for 15 epochs, in shuffled groups of 16 frames. The loss is the sum of the cross-entropy and the soft Dice loss, which is one minus the Dice coefficient computed from the probabilities of the cavity. Use a one-cycle learning rate with a highest value of 0.003. After every epoch, compute the mean Dice coefficient of the validation frames, and keep the weights of the epoch where it is highest. Make the random choices the same in every run by using the number 2026 as the starting value. Then threshold the probability of the cavity in the test frames at 0.5 and compute the Dice coefficient of every test frame against its expert outline. Report its mean by view and phase and by image quality.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table of the mean Dice coefficient by view and phase and a second table by image quality, and keep the mean Dice coefficient of the 200 test frames, with three decimals, under the name "answer".  
> Test: Each of the four groups of view and phase must hold 50 test frames, and the mean Dice coefficient of the 200 test frames must lie between 0.85 and 0.97.  

In [ ]:
#@title Data of project step 1: run this cell (it prepares the table "data" and shows it)
_FSETS = {"training": "training", "validation": "validation", "testing": "test"}
data = pd.DataFrame({"patient": frame_patient, "view": frame_view, "phase": frame_phase,
                     "set": [_FSETS[s] for s in frame_split],
                     "image quality": [_quality[(p, v)] for p, v in zip(frame_patient, frame_view)],
                     "frame": list(images), "cavity": list((masks == 1).astype(np.uint8))})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: Volumes, EF and Bland-Altman

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Before automatic volumes and an automatic ejection fraction are used, the cardiologist wants to compare all three with the expert on the 50 test patients of CAMUS.  
> Data: The notebook holds a table named "data" with the 50 test patients of CAMUS. Each row is a patient: "patient" gives the patient; "expert EDV" and "network EDV" give the end-diastolic volume and "expert ESV" and "network ESV" the end-systolic volume in mL; "expert EF" and "network EF" give the ejection fraction in percent. The values of the expert come from the outlines of the expert, those of the network from the outlines drawn by a U-Net, all by the biplane method of discs.  
> Task: For each of the three measurements, EDV, ESV and the EF, compute the difference for every patient, network minus expert, then the mean difference, the standard deviation of the differences and the limits of agreement, the mean ± 1.96 standard deviations. Draw a Bland-Altman plot for each measurement: the difference against the mean of the two values, with horizontal lines at the mean difference and at the two limits.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row per measurement and the columns mean difference, standard deviation, lower limit and upper limit, and the three Bland-Altman plots; keep the mean difference of the EF, with two decimals, under the name "answer".  
> Test: There must be 50 patients, and the expert EF of patient0201 must be 42.1 percent.  

In [ ]:
#@title Data of project step 2: run this cell (it prepares the table "data" and shows it)
_a = _course_agreement()
data = pd.DataFrame({"patient": _a["patient"],
                     "expert EDV": _a["edv_expert"], "network EDV": _a["edv_net"],
                     "expert ESV": _a["esv_expert"], "network ESV": _a["esv_net"],
                     "expert EF": _a["ef_expert"], "network EF": _a["ef_net"]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: The ranges of EF

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know for how many of the 50 test patients the network and the expert place the ejection fraction in the same clinical range.  
> Data: The notebook holds a table named "data" with the 50 test patients of CAMUS. Each row is a patient: "patient" gives the patient; "expert EDV" and "network EDV" give the end-diastolic volume and "expert ESV" and "network ESV" the end-systolic volume in mL; "expert EF" and "network EF" give the ejection fraction in percent. The values of the expert come from the outlines of the expert, those of the network from the outlines drawn by a U-Net, all by the biplane method of discs.  
> Task: Sort each EF, that of the expert and that of the network, into three ranges: 40 percent or less, above 40 and below 50 percent, and 50 percent or more. Make a cross table with the range of the expert in the rows and the range of the network in the columns, in the order of the ranges, and count the patients on its diagonal.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the cross table, and keep the number of patients on its diagonal under the name "answer".  
> Test: The cross table must add up to 50 patients.  

In [ ]:
#@title Data of project step 3: run this cell (it prepares the table "data" and shows it)
_a = _course_agreement()
data = pd.DataFrame({"patient": _a["patient"],
                     "expert EDV": _a["edv_expert"], "network EDV": _a["edv_net"],
                     "expert ESV": _a["esv_expert"], "network ESV": _a["esv_net"],
                     "expert EF": _a["ef_expert"], "network EF": _a["ef_net"]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: The largest errors

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: For the test patients whose automatic ejection fraction differs most from the expert, the cardiologist wants to see the frames and the two outlines, to judge where the network went wrong.  
> Data: The notebook holds a table named "data" with the 200 test frames of CAMUS, four for each of the 50 test patients. Each row is a frame: "patient" gives the patient, "view" the view ("2CH" or "4CH"), "phase" the phase ("ED" or "ES") and "image quality" the quality of the view ("good", "medium" or "poor"). Each cell of the column "frame" holds the frame as an image of 96 × 96 pixels; "expert cavity" and "network cavity" hold the cavity of the left ventricle at the same size as drawn by the expert and by a U-Net, 1 inside the cavity and 0 outside. "expert EF" and "network EF" give the ejection fraction of the patient in percent, the same on the four rows of a patient.  
> Task: Find the three patients with the largest absolute difference between the two EFs. For each of them, show its four frames with the outline of the expert cavity and that of the network cavity drawn on the frame in two colours, and write the two EFs and the image quality in the title.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show one figure of four frames for each of the three patients, and keep the largest absolute difference of the EF, in percentage points with one decimal, under the name "answer".  
> Test: The three patients shown must be those with the three largest absolute differences of the EF, and each figure must hold four frames.  

In [ ]:
#@title Data of project step 4: run this cell (it prepares the table "data" and shows it)
_a = _course_agreement().set_index("patient")
_p, _te = _course_unet(), np.where(frame_split == "testing")[0]
data = pd.DataFrame({"patient": frame_patient[_te], "view": frame_view[_te], "phase": frame_phase[_te],
                     "image quality": [_quality[(frame_patient[i], frame_view[i])] for i in _te],
                     "frame": list(images[_te]), "expert cavity": list((masks[_te] == 1).astype(np.uint8)),
                     "network cavity": list((_p > 0.5).astype(np.uint8)),
                     "expert EF": [_a.loc[frame_patient[i], "ef_expert"] for i in _te],
                     "network EF": [_a.loc[frame_patient[i], "ef_net"] for i in _te]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

<a id="part-2"></a>

# Part 2: Python path

**For whom?** This part is for participants who want to learn to write Python. Part 1 is not needed for it: After Section 0, start here.

**How it works.** Run the cells from top to bottom with Shift+Enter. The *Python step* at the start teaches the parts of the language that the week needs. In the numbered sections, exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. Cells marked *Advanced (optional)* are extensions for those who already program. A participant who opens this part in the middle runs the *Catch-up* cell at the start of that section. Some exercise cells begin with a line that prepares a table named `data`: It serves the prompt of the exercise in the prompt workshop of the interactive lab, and on the Python path the exercise can be solved with the tables named in its text.

## Python step 4: Reusing trained models

This week the code reuses work that has already been done: a network trained on more than a million photographs, and expert outlines drawn on echocardiograms. This step reads one frame of CAMUS with its outline, turns the outline into an array of true and false values and measures the area of the left ventricle in square centimetres. It then loads the ImageNet weights of EfficientNet-B0, freezes its layers and counts the weights that are still trained.

In [ ]:
# Python practice: a small example for learning the language
img, gt, spacing = camus.load("patient0201", "4CH", "ED")
print(img.shape, gt.shape, spacing)
print("labels in the outline:", np.unique(gt))
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.imshow(img, cmap="gray")
ax.contour(gt == 1, levels=[0.5], colors="#E5797B", linewidths=1.2)
ax.set_title("Patient 201, four-chamber view, end-diastole", fontsize=9)
ax.axis("off")
plt.show()

**Python note.** `camus.load` returns three things, and `img, gt, spacing = ...` gives each a name in one line; this is called *unpacking*. `img` holds the grey values of the frame, `gt` one label per pixel (0 background, 1 cavity of the left ventricle, 2 myocardium, 3 left atrium) and `spacing` the size of a pixel in mm, as (height, width). `np.unique(gt)` lists the different values of an array once each. The frame is shown in the usual orientation, with the apex at the top. Images of CAMUS: CAMUS, CC BY-NC-SA 4.0.

In [ ]:
# Python practice: a mask and a function that measures it
cavity = gt == 1                      # True inside the cavity, False elsewhere
print(cavity.dtype, cavity.sum(), "pixels in the cavity")


def area_cm2(mask, spacing):
    """Area of a mask in cm²: number of pixels × area of one pixel."""
    return float(mask.sum() * spacing[0] * spacing[1] / 100)


print("Area of the cavity:", round(area_cm2(cavity, spacing), 1), "cm²")

**Python note.** `gt == 1` compares every pixel with 1 and gives an array of `True` and `False`, a *mask*. `.sum()` counts the `True` values, because Python counts `True` as 1. A pixel of 0.308 × 0.308 mm covers 0.095 mm², and 100 mm² make 1 cm². `def area_cm2(mask, spacing):` defines a function that receives two values and gives back one with `return`; the text between triple quotes under it describes the function.

In [ ]:
# Python practice: a trained network, frozen
net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
print("All weights:", sum(p.numel() for p in net.parameters()))
print(net.classifier)
for p in net.parameters():
    p.requires_grad = False                                     # frozen
net.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))  # a new last layer for three groups
print("Weights still trained:", sum(p.numel() for p in net.parameters() if p.requires_grad))

**Python note.** `efficientnet_b0(weights=...)` builds the network and loads the weights learned on ImageNet; the weights were downloaded in Section 0. Every weight of a PyTorch network has the attribute `requires_grad`: When it is `False`, autograd computes no gradient for it and the optimiser leaves it unchanged, so the layer is *frozen*. A new layer, created after the freezing, starts with `requires_grad = True`. `sum(... for p in ... if ...)` adds up only the elements that pass the condition.

**Quick check.** A frozen network with a new last layer of 1280 inputs and 3 outputs is trained. Which weights change?

<details><summary>Answer</summary>

Only the 1280 × 3 weights and the 3 biases of the new last layer, 3843 numbers; the four million weights learned on ImageNet stay as they are.

</details>

**Exercise 4.1.** How many pixels of the four-chamber end-diastolic frame of patient 201 belong to the cavity of the left ventricle?

In [ ]:
data = exercise_table("4.1")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 4.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int((gt == 1).sum())
teach.check("Exercise 4.1", answer)

**Exercise 4.2.** What is the area of the cavity of the left ventricle in the four-chamber end-systolic frame of patient 201, in cm²? Round to one decimal.

In [ ]:
data = exercise_table("4.2")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 4.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
img_es, gt_es, spacing_es = camus.load("patient0201", "4CH", "ES")
answer = round(area_cm2(gt_es == 1, spacing_es), 1)
teach.check("Exercise 4.2", answer)

**Exercise 4.3.** In `net`, all weights of EfficientNet-B0 are frozen and the new last layer has three outputs. How many weights are still trained?

In [ ]:
answer = None   # your answer
teach.check("Exercise 4.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in net.parameters() if p.requires_grad)
teach.check("Exercise 4.3", answer)

## 1. What a network learned from photographs

*Lecture: [Transfer learning: What a network learned from photographs](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#transfer-learning-what-a-network-learned-from-photographs)*

EfficientNet-B0 learned to name 1000 kinds of objects from about 1.2 million photographs of ImageNet. Its first layer is a set of small filters over the three colour channels; deeper layers combine them into textures and parts of objects. A slice of cine cardiac magnetic resonance has no colours, so the course turns the end-diastolic frame, the end-systolic frame and their mean into three channels, and normalises them as the photographs of ImageNet were normalised.

**From the formulas to the code.** The normalisation of channel $c$ is $x' = (x - \mu_c)/\sigma_c$, with $\mu = (0.485, 0.456, 0.406)$ and $\sigma = (0.229, 0.224, 0.225)$: `(x3 - IMAGENET_MEAN) / IMAGENET_SD`. `F.interpolate(..., size=(128, 128))` enlarges the slices of 64 × 64 pixels. `net.features[0][0].weight` holds the 32 filters of the first layer, each of size 3 × 3 over 3 channels.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(1)

In [ ]:
# The 32 filters of the first layer, shown as small colour images
imagenet_net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).eval()
filters = imagenet_net.features[0][0].weight.detach()      # (32, 3, 3, 3)
print(tuple(filters.shape))
fig, axes = plt.subplots(4, 8, figsize=(6.4, 3.4))
for k, ax in enumerate(axes.flat):
    f = filters[k].permute(1, 2, 0).numpy()
    ax.imshow((f - f.min()) / (f.max() - f.min() + 1e-9))
    ax.axis("off")
fig.suptitle("First layer of EfficientNet-B0, learned on ImageNet", fontsize=9)
plt.show()

**Python note.** `.detach()` takes the numbers out of autograd, so that they can be shown. A filter has the shape (channels, rows, columns), while `imshow` expects (rows, columns, channels); `.permute(1, 2, 0)` reorders the dimensions. Each filter is stretched to the range 0 to 1 before it is shown. Many filters look like small edges or contrasts of colour: These are the first things a network learns from photographs, and they are just as useful for the border of a ventricle.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_SD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def as_photos(x, size=128):
    """ED, ES and their mean as three colour channels, resized and normalised like ImageNet."""
    x = torch.as_tensor(x, dtype=torch.float32)
    x3 = torch.stack([x[:, 0], x[:, 1], (x[:, 0] + x[:, 1]) / 2], dim=1)
    x3 = F.interpolate(x3, size=(size, size), mode="bilinear", align_corners=False)
    return (x3 - IMAGENET_MEAN) / IMAGENET_SD

P = acdc.pairs(GROUPS)
X, y, patients, sets, classes = P["X"], P["y"], P["patient"], P["set"], P["classes"]
photos = as_photos(X)
print(tuple(X.shape), "→", tuple(photos.shape), classes)

**Python note.** `acdc.pairs` is the tool of Week 3: the three middle slices of the 90 patients, each with two frames of 64 × 64 pixels, and the set of every slice (45 training, 15 validation and 30 test patients). `torch.stack(..., dim=1)` puts three images side by side as channels. `.view(1, 3, 1, 1)` shapes the three means so that each is subtracted from its own channel in every pixel.

In [ ]:
# One slice of patient 134 as ImageNet sees it
k = int(np.where(patients == "patient134")[0][1])
rgb = photos[k] * IMAGENET_SD[0] + IMAGENET_MEAN[0]
fig, axes = plt.subplots(1, 4, figsize=(8.4, 2.4))
for ax, image, title in zip(axes, [rgb[0], rgb[1], rgb[2], rgb.permute(1, 2, 0)],
                            ["red: ED", "green: ES", "blue: mean", "together"]):
    ax.imshow(image.clamp(0, 1).numpy(), cmap="gray" if image.ndim == 2 else None)
    ax.set_title(title, fontsize=8)
    ax.axis("off")
fig.suptitle("ACDC, CC BY-NC-SA 4.0", fontsize=7, y=0.04)
plt.show()

**Observe and reflect.** In the image on the right, the structures that do not move appear grey, because the three channels agree there, and colour appears where the end-diastolic and the end-systolic frames differ. Where in the slice would you expect colour in a patient with dilated cardiomyopathy and a very low ejection fraction?

**Exercise A.** How many weights does EfficientNet-B0 have as it was trained on ImageNet, with its last layer of 1000 classes?

In [ ]:
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in imagenet_net.parameters())
teach.check("Exercise A", answer)

## 2. Feature extraction: a frozen network and a simple model

*Lecture: [Transfer learning: What a network learned from photographs](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#transfer-learning-what-a-network-learned-from-photographs)*

In feature extraction, the network trained on ImageNet is not trained again. Every slice passes once through its frozen layers, and the average pooling before the last layer describes the slice with 1280 numbers, the features. A simple model, here a logistic regression, learns the three groups from these numbers. The decision for a patient is the group with the highest mean probability over the patient's three slices, as in Week 3.

**From the formulas to the code.** The features of a slice are $\mathbf{f} = \text{pool}(\text{features}(x)) \in \mathbb{R}^{1280}$: `net.avgpool(net.features(x)).flatten(1)`. The logistic regression of Week 2, extended to three classes, gives $p_k = e^{z_k}/\sum_j e^{z_j}$ with $z_k = \mathbf{w}_k \cdot \mathbf{f} + b_k$: `make_pipeline(StandardScaler(), LogisticRegression(C=0.1))`, where a small `C` keeps the weights small with 1280 features and few patients.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(2)

In [ ]:
def by_patient(prob, mask):
    """Mean probabilities over the slices of each patient, and the share of correct decisions."""
    names = sorted(set(patients[mask]))
    mean_prob = np.array([prob[patients[mask] == n].mean(axis=0) for n in names])
    truth = np.array([y[mask][patients[mask] == n][0] for n in names])
    return float((mean_prob.argmax(axis=1) == truth).mean())

with torch.no_grad():
    features = torch.cat([imagenet_net.avgpool(imagenet_net.features(photos[i:i + 32])).flatten(1)
                          for i in range(0, len(photos), 32)]).numpy()
tr, va, te = sets == "train", sets == "val", sets == "test"
feature_model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000))
feature_model.fit(features[tr], y[tr])
acc_features = round(by_patient(feature_model.predict_proba(features[te]), te), 3)
print(features.shape, "test accuracy:", acc_features)

**Python note.** `with torch.no_grad():` switches off the recording of autograd, because no weight is trained here; this saves memory and time. The slices pass through the network in groups of 32, `photos[i:i + 32]`, and `torch.cat` joins the 9 groups again. `by_patient` averages the probabilities of the three slices of each patient, as in Week 3, and gives the share of correct decisions. Computing the features of 270 slices takes a few seconds.

In [ ]:
# The decisions of the 30 test patients
names = sorted(set(patients[te]))
prob_te = feature_model.predict_proba(features[te])
decision = [int(prob_te[patients[te] == n].mean(axis=0).argmax()) for n in names]
truth = [int(y[patients == n][0]) for n in names]
print(pd.DataFrame(confusion_matrix(truth, decision), index=classes, columns=classes))

**Observe and reflect.** Which group does the logistic regression confuse most often? The features were learned on photographs of animals, vehicles and objects. Why can they still separate a dilated from a thick-walled ventricle?

**Exercise B.** What is the accuracy of the same logistic regression on the 45 training patients, with one decision per patient? Round to three decimals. Compare it with the test accuracy.

In [ ]:
data = exercise_table("B")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(by_patient(feature_model.predict_proba(features[tr]), tr), 3)
teach.check("Exercise B", answer)

## 3. Fine-tuning: every layer adapts

*Lecture: [Transfer learning: What a network learned from photographs](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#transfer-learning-what-a-network-learned-from-photographs)*

In fine-tuning, every weight of EfficientNet-B0 starts from ImageNet and keeps learning from the slices, with a small learning rate. The new last layer, with dropout and three outputs, starts from random weights. The training loop is the loop of Week 3: It uses mini-batches, turns the slices by a quarter turn as data augmentation, measures the validation loss after every epoch and keeps the weights of the best epoch.

**From the formulas to the code.** Each update moves all four million weights against their gradients, $w \leftarrow w - \eta\,\partial L/\partial w$ in its simplest form. Adam scales each step with running averages of the gradients, here with the learning rate $\eta = 0.001$: `torch.optim.Adam(net.parameters(), lr=1e-3)`. `net.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))` replaces the last layer of 1000 classes. `fine_tune` returns the training and validation loss of every epoch.

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(3)

In [ ]:
def fine_tune(net, x_tr, y_tr, x_va, y_va, epochs=15, lr=1e-3, batch=16, seed=RANDOM_STATE):
    """Train every weight of net; keep the epoch with the lowest validation loss."""
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-4)
    order_gen = torch.Generator().manual_seed(seed)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        net.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            turns = int(torch.randint(0, 4, (1,), generator=order_gen))
            loss = F.cross_entropy(net(torch.rot90(x_tr[j], turns, dims=(2, 3))), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        net.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(net(x_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(net.state_dict())
    net.load_state_dict(best_state)
    net.eval()
    return history

y_t = torch.tensor(y, dtype=torch.long)
torch.manual_seed(RANDOM_STATE)
tuned = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
tuned.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))
history = fine_tune(tuned, photos[tr], y_t[tr], photos[va], y_t[va])
with torch.no_grad():
    prob_tuned = F.softmax(tuned(photos[te]), dim=1).numpy()
acc_finetune = round(by_patient(prob_tuned, te), 3)
print("Fine-tuned, test accuracy:", acc_finetune, "· features:", acc_features)

**Python note.** The function `fine_tune` gathers the training loop of Week 3, so that the next sections can call it again with other patients. `torch.randint(0, 4, (1,), generator=order_gen)` draws the number of quarter turns from the same random generator as the order of the slices, so the same seed gives the same training. `copy.deepcopy(net.state_dict())` keeps a full copy of the best weights. With 135 training slices this cell takes about a minute on a laptop.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(range(1, len(history) + 1), [h[0] for h in history], color="#1F5F8B", label="training")
ax.plot(range(1, len(history) + 1), [h[1] for h in history], color="#9B1B1B", label="validation")
ax.set_xlabel("epoch")
ax.set_ylabel("cross-entropy loss")
ax.legend()
plt.show()

**Observe and reflect.** The run of the course reached 0.967 with fine-tuning and 0.867 with features alone, on the same 30 test patients. One patient more or less moves the accuracy by 0.033. What would you need before you call one method better than the other?

**Advanced (optional).** The same network can also start from random weights: `efficientnet_b0(weights=None)`. The cell below trains it exactly like the fine-tuned network, which takes about another minute. In the run of the course it reached 0.833 with 15 patients per group and stayed at 0.333, the share of one group, with five patients per group.

In [ ]:
torch.manual_seed(RANDOM_STATE)
from_random = efficientnet_b0(weights=None)
from_random.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))
fine_tune(from_random, photos[tr], y_t[tr], photos[va], y_t[va])
with torch.no_grad():
    acc_random = round(by_patient(F.softmax(from_random(photos[te]), dim=1).numpy(), te), 3)
print("From random weights, test accuracy:", acc_random)

**Exercise C.** Fine-tuning runs 15 epochs over the 135 training slices in mini-batches of 16 slices; the last mini-batch of an epoch is smaller. How many times are the weights updated in total?

In [ ]:
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = math.ceil(135 / 16) * 15
teach.check("Exercise C", answer)

## 4. A fair comparison with few patients

*Lecture: [Learning from few labelled patients](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#learning-from-few-labelled-patients)*

A result from few patients depends on which patients were chosen. A fair comparison therefore draws k training patients per group several times, gives every method the same draws and the same validation and test patients, and reports the mean together with the spread. This section repeats the draws of the course for the method of features, which takes seconds, and places it next to the three other methods of the lecture page, whose training took the course about seventeen minutes.

**From the formulas to the code.** For k patients per group and draw $d$, the accuracy is $a_{k,d}$; the curve shows $\bar{a}_k = \frac{1}{D}\sum_d a_{k,d}$ with its smallest and largest value: `np.mean(curve[k])`, `min(curve[k])` and `max(curve[k])`. The draws come from `np.random.default_rng(RANDOM_STATE)` in a fixed order: k = 1, 2, 5, 10 and 15, five draws for every k below 15.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(4)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
train_names = sorted(set(patients[tr]))
group_of = {n: int(y[patients == n][0]) for n in train_names}
curve = {}
for k in (1, 2, 5, 10, 15):
    curve[k] = []
    for draw in range(5 if k < 15 else 1):
        chosen = []
        for g in range(3):
            pool = [n for n in train_names if group_of[n] == g]
            chosen += sorted(rng.choice(pool, size=k, replace=False).tolist())
        rows = np.isin(patients, chosen)
        small = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000))
        small.fit(features[rows], y[rows])
        curve[k].append(by_patient(small.predict_proba(features[te]), te))
for k, accs in curve.items():
    print(k, "per group:", np.round(accs, 3), "mean", round(float(np.mean(accs)), 3))

**Python note.** `rng.choice(pool, size=k, replace=False)` draws k different names from a list; `replace=False` forbids drawing the same patient twice. `np.isin(patients, chosen)` marks the slices of the chosen patients. `range(5 if k < 15 else 1)` repeats five times, except for k = 15: The training set of ACDC holds exactly 15 patients per group, so there is only one possible draw.

In [ ]:
# The three other methods: mean accuracy over the same draws, from the run of the course
COURSE_RUN = {"network of Week 3 from scratch": [0.480, 0.573, 0.720, 0.820, 0.933],
              "EfficientNet-B0 from random weights": [0.333, 0.333, 0.333, 0.740, 0.833],
              "EfficientNet-B0 fine-tuned": [0.480, 0.607, 0.793, 0.860, 0.967]}
ks = list(curve)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.fill_between(ks, [min(curve[k]) for k in ks], [max(curve[k]) for k in ks], color="#B97813", alpha=0.2)
ax.plot(ks, [np.mean(curve[k]) for k in ks], "o-", color="#B97813", label="ImageNet features (this notebook)")
for (name, accs), color in zip(COURSE_RUN.items(), ["#566A73", "#888888", "#1F5F8B"]):
    ax.plot(ks, accs, "s--", color=color, label=name, lw=1)
ax.set_xlabel("training patients per group")
ax.set_ylabel("accuracy, 30 test patients")
ax.set_ylim(0.2, 1.02)
ax.legend(fontsize=7)
plt.show()

**Observe and reflect.** With one or two patients per group, the band of the features is wide: The same method gives very different results with different patients. Which method would you choose for a new laboratory that has labelled ten examinations per group, and what would you report next to its mean accuracy?

**Exercise D.** What is the mean accuracy of the method of features over the five draws with five patients per group? Round to three decimals.

In [ ]:
data = exercise_table("D")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(np.mean(curve[5])), 3)
teach.check("Exercise D", answer)

## 5. CAMUS: frames and outlines as arrays

*Lecture: [Segmentation: A label for every pixel](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#segmentation-a-label-for-every-pixel)*

CAMUS holds the apical two-chamber and four-chamber views of 500 patients of the University Hospital of Saint-Étienne. For each view, the end-diastolic and the end-systolic frame come with expert outlines. The course uses 300 patients: 200 of the training set of the authors, their 50 validation patients and their 50 test patients. For the network, every frame is resized to 96 × 96 pixels; the outlines are resized with the nearest label, so that no new labels appear between two classes.

**From the formulas to the code.** A frame is an array $x \in [0, 1]^{96 \times 96}$ and its outline an array of labels $m \in \{0, 1, 2, 3\}^{96 \times 96}$: `images[i]` and `masks[i]`. The target of the network is $t = [m = 1]$, the cavity of the left ventricle: `masks == 1`. `camus.frames` gives 1200 frames, four per patient, with their patient, view, phase and set.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(5)

In [ ]:
frames_ = camus.frames(camus.patients(), size=96)
images, masks = frames_["images"], frames_["masks"]
frame_patient, frame_view, frame_phase, frame_split = frames_["patient"], frames_["view"], frames_["phase"], frames_["split"]
frame_shape = frames_["shape"]
camus_table = camus.table()
print(images.shape, masks.shape, {s: int((frame_split == s).sum()) for s in ("training", "validation", "testing")})
print(camus_table.set_index("patient").loc["patient0201"])

**Python note.** `camus.frames` reads the four frames of every patient from the files downloaded in Section 0 and resizes them; reading 1200 frames takes about fifteen seconds. `camus.table` reads the Info files: sex, age, the image quality of both views and the ejection fraction (EF) that the authors computed from the expert outlines. `.loc["patient0201"]` takes the row of the case of the week.

In [ ]:
# The four frames of patient 201 at their original size, with the expert outlines
fig, axes = plt.subplots(1, 4, figsize=(9.6, 3.2))
for ax, (view, phase) in zip(axes, [("2CH", "ED"), ("2CH", "ES"), ("4CH", "ED"), ("4CH", "ES")]):
    im, lab, _ = camus.load("patient0201", view, phase)
    ax.imshow(im, cmap="gray")
    for label, color in ((1, "#E5797B"), (2, "#F9AB00"), (3, "#7DB4DD")):
        ax.contour(lab == label, levels=[0.5], colors=color, linewidths=1)
    ax.set_title(f"{view} {phase}", fontsize=9)
    ax.axis("off")
fig.suptitle("red: LV cavity · yellow: myocardium · blue: left atrium · CAMUS, CC BY-NC-SA 4.0", fontsize=8, y=0.06)
plt.show()

**Observe and reflect.** The end-systolic cavity is smaller and its border closer to the myocardium. The two views cut the ventricle in two planes about 60 degrees apart. Why does the method of discs need both views, and not one?

**Exercise E.** What percentage of all pixels of the 800 training frames, at 96 × 96 pixels, belongs to the cavity of the left ventricle? Round to one decimal.

In [ ]:
data = exercise_table("E")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(100 * float((masks[frame_split == "training"] == 1).mean()), 1)
teach.check("Exercise E", answer)

## 6. The Dice coefficient

*Lecture: [Segmentation: A label for every pixel](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#segmentation-a-label-for-every-pixel)*

The cavity covers less than a tenth of the pixels, so a network that calls every pixel background is right on more than 90 percent of them and still useless. The Dice coefficient compares two outlines directly: twice their overlap divided by the sum of their sizes. It is 1 for identical outlines and 0 for outlines without a common pixel.

**From the formulas to the code.** $D = 2|A \cap B|/(|A| + |B|)$: `2 * (a & b).sum() / (a.sum() + b.sum())`. `a & b` is true where both masks are true. `np.roll(a, 3, axis=1)` shifts a mask by 3 columns to the right, as if the border were drawn 3 pixels too far.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(6)

In [ ]:
def dice(pred, true):
    """Dice coefficient of each frame: two boolean arrays of the same shape."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


def frame_of(patient, view, phase):
    return int(np.where((frame_patient == patient) & (frame_view == view) & (frame_phase == phase))[0][0])


rows = []
for phase in ("ED", "ES"):
    a = masks[frame_of("patient0201", "4CH", phase)] == 1
    for shift in range(7):
        rows.append(dict(phase=phase, shift=shift, dice=round(float(dice(np.roll(a, shift, axis=1), a)), 3)))
shifts = pd.DataFrame(rows).pivot(index="shift", columns="phase", values="dice")
print(shifts)

**Python note.** `axis=(-2, -1)` sums over the last two dimensions, rows and columns, so the same function gives one Dice coefficient for one frame or one per frame for a stack of frames. `np.maximum(..., 1)` avoids a division by zero when both masks are empty. `.pivot` turns the long table of 14 rows into one row per shift and one column per phase.

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.2))
for phase, color in (("ED", "#1F5F8B"), ("ES", "#9B1B1B")):
    ax.plot(shifts.index, shifts[phase], "o-", color=color, label=phase)
ax.set_xlabel("shift of the outline (pixels of 96)")
ax.set_ylabel("Dice")
ax.legend()
plt.show()

**Observe and reflect.** The same shift costs more at end-systole than at end-diastole. A network is judged by its mean Dice coefficient over all frames. Why should a laboratory also ask for the Dice coefficient of the end-systolic frames alone?

**Exercise F.** What is the Dice coefficient between the cavity of the four-chamber view of patient 201 at end-diastole and at end-systole, both at their original size? Round to three decimals.

In [ ]:
data = exercise_table("F")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
_, lab_ed, _ = camus.load("patient0201", "4CH", "ED")
_, lab_es, _ = camus.load("patient0201", "4CH", "ES")
answer = round(float(dice(lab_ed == 1, lab_es == 1)), 3)
teach.check("Exercise F", answer)

## 7. A U-Net for the cavity of the left ventricle

*Lecture: [A U-Net on the echocardiograms of CAMUS](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#a-u-net-on-the-echocardiograms-of-camus)*

A U-Net has the shape of its name. Its encoder shrinks the frame from 96 to 12 pixels in three steps and grows the number of channels from 8 to 64, so that every pixel at the bottom sees a large part of the heart. Its decoder enlarges the frame again in three steps, and at every level a skip connection hands over the fine detail of the encoder. The last layer gives two numbers per pixel, background and cavity. The loss adds the cross-entropy of every pixel and the soft Dice loss, which weighs the small cavity as much as the large background.

**From the formulas to the code.** The soft Dice loss is $L_D = 1 - \frac{2\sum_j p_j t_j + 1}{\sum_j p_j + \sum_j t_j + 1}$, with the probability $p_j$ of the cavity and the target $t_j$ of pixel $j$: `soft_dice_loss`. The loss of a mini-batch is $L = L_{CE} + L_D$: `F.cross_entropy(out, target) + soft_dice_loss(out, target)`. `torch.cat([self.up3(b), d3], dim=1)` is a skip connection: The enlarged map and the map of the encoder are stacked as channels. `OneCycleLR` raises the learning rate to 0.003 and lowers it again over the 15 epochs.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(7)

In [ ]:
def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Three levels down, a bottleneck and three levels up with skip connections."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Dice coefficient of each frame: two boolean arrays of the same shape."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)

print("Trainable parameters of the U-Net:", sum(p.numel() for p in UNet().parameters()))

**Python note.** `class UNet(nn.Module)` follows the pattern of the network of Week 3: `__init__` creates the layers and `forward` says how a frame passes through them. The function `unet_block` builds the two convolutions of one level, so that the class can reuse it seven times. `nn.ConvTranspose2d(..., 2, stride=2)` doubles the size of a map. `d1`, `d2` and `d3` keep the maps of the encoder until the decoder needs them.

In [ ]:
cavity = masks == 1
tr, va, te = frame_split == "training", frame_split == "validation", frame_split == "testing"
X_img = torch.tensor(images)[:, None]
Y_cav = torch.tensor(cavity.astype(np.int64))
torch.manual_seed(RANDOM_STATE)
unet = UNet()
epochs, batch = 15, 16
optimizer = torch.optim.Adam(unet.parameters(), lr=3e-3)
steps = epochs * ((int(tr.sum()) + batch - 1) // batch)
schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=steps)
order_gen = torch.Generator().manual_seed(RANDOM_STATE)
X_tr, Y_tr = X_img[tr], Y_cav[tr]
best, best_state = -1.0, None
for epoch in range(epochs):
    unet.train()
    order = torch.randperm(len(X_tr), generator=order_gen)
    for start in range(0, len(X_tr), batch):
        j = order[start:start + batch]
        out = unet(X_tr[j])
        loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        schedule.step()
    unet.eval()
    with torch.no_grad():
        val_dice = float(dice(unet(X_img[va]).argmax(dim=1).numpy() == 1, cavity[va]).mean())
    if val_dice > best:
        best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
unet.load_state_dict(best_state)
unet.eval()
with torch.no_grad():
    test_prob = F.softmax(unet(X_img[te]), dim=1)[:, 1].numpy()

test_dice = round(float(dice(test_prob > 0.5, cavity[te]).mean()), 3)
print("Best epoch:", best_epoch, "· Dice of the 200 test frames:", test_dice)

**Python note.** The loop is that of Week 3, with two changes. The scheduler `OneCycleLR` changes the learning rate after every mini-batch, and the best epoch is chosen by the Dice coefficient of the validation frames instead of their loss. `unet(...).argmax(dim=1)` gives the class with the larger output in every pixel. Training takes about three minutes on a laptop; in the run of the course the best epoch was 13 and the test frames reached a mean Dice coefficient of 0.924.

In [ ]:
per_frame = dice(test_prob > 0.5, cavity[te])
quality = {}
for _, r in camus_table.iterrows():
    quality[(r["patient"], "2CH")], quality[(r["patient"], "4CH")] = r["quality_2ch"], r["quality_4ch"]
test_frames = pd.DataFrame({"patient": frame_patient[te], "view": frame_view[te], "phase": frame_phase[te], "dice": per_frame})
test_frames["quality"] = [quality[(p, v)] for p, v in zip(test_frames["patient"], test_frames["view"])]
print(test_frames.groupby(["view", "phase"])["dice"].mean().round(3).unstack())
print(test_frames.groupby("quality")["dice"].agg(["count", "mean"]).round(3))

**Observe and reflect.** The two-chamber view at end-systole is the weakest group, and the frames of poor quality draw worse than the good ones. Which of these two findings would you report to a laboratory that wants to use the network, and in which words?

In [ ]:
# The case of the week: expert (red) and network (blue) at the original size
test_index = np.where(te)[0]
net_masks = {}
for k, i in enumerate(test_index):
    net_masks.setdefault(frame_patient[i], {})[(frame_view[i], frame_phase[i])] = camus.to_original(test_prob[k], frame_shape[i])
fig, axes = plt.subplots(1, 4, figsize=(9.6, 3.2))
for ax, key in zip(axes, [("2CH", "ED"), ("2CH", "ES"), ("4CH", "ED"), ("4CH", "ES")]):
    im, lab, _ = camus.load("patient0201", *key)
    ax.imshow(im, cmap="gray")
    ax.contour(lab == 1, levels=[0.5], colors="#E5797B", linewidths=1.2)
    ax.contour(net_masks["patient0201"][key], levels=[0.5], colors="#7DB4DD", linewidths=1.2)
    ax.set_title(f"{key[0]} {key[1]}: Dice {dice(net_masks['patient0201'][key].astype(bool), lab == 1):.3f}", fontsize=8)
    ax.axis("off")
fig.suptitle("CAMUS, CC BY-NC-SA 4.0", fontsize=7, y=0.06)
plt.show()

**Python note.** `camus.to_original` enlarges the probabilities of the network, not its labels, to the original frame of about 520 × 630 pixels, thresholds them at 0.5 and keeps the largest connected region with its holes filled. Enlarging the probabilities gives a smooth border; enlarging the labels would give steps of five pixels. `net_masks` is a dictionary of dictionaries: one entry per patient, and in it one mask per pair of view and phase.

**Exercise G.** How many trainable parameters does a U-Net with 16 instead of 8 channels at its first level have, `UNet(c=16)`?

In [ ]:
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in UNet(c=16).parameters() if p.requires_grad)
teach.check("Exercise G", answer)

## 8. Volumes and the ejection fraction by the method of discs

*Lecture: [The ejection fraction by the method of discs](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#the-ejection-fraction-by-the-method-of-discs)*

The biplane method of discs, the modified Simpson's rule of the guidelines, cuts the ventricle into 20 elliptic discs along its long axis, from the mitral annulus to the apex. The two-chamber view gives one diameter of each disc and the four-chamber view the other. The thickness of a disc is a twentieth of the long axis; in the reference script of CAMUS, both volumes use the longest of the four outlines.

**From the formulas to the code.** $V = \frac{\pi}{4}\sum_{i=1}^{20} a_i\,b_i\,\frac{L}{20}$, in mm³, divided by 1000 for mL: `np.pi / 4 * np.sum(a * b) * step / 1000`. $EF = 100\,(EDV - ESV)/EDV$: `camus.ejection_fraction(edv, esv)`. `camus.discs(mask, spacing)` returns the 20 diameters in mm, the thickness of a disc in mm, the middle of the annulus and the apex.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(8)

In [ ]:
outline, discs_of = {}, {}
for view in ("2CH", "4CH"):
    for phase in ("ED", "ES"):
        _, lab, sp = camus.load("patient0201", view, phase)
        outline[(view, phase)] = (lab == 1, sp)
        discs_of[(view, phase)] = camus.discs(lab == 1, sp)
        d, step, base, apex = discs_of[(view, phase)]
        print(view, phase, "L =", round(step * 20, 1), "mm · thickness", round(step, 2), "mm · widest disc", round(d.max(), 1), "mm")
step = max(x[1] for x in discs_of.values())
edv = np.pi / 4 * np.sum(discs_of[("2CH", "ED")][0] * discs_of[("4CH", "ED")][0]) * step / 1000
esv = np.pi / 4 * np.sum(discs_of[("2CH", "ES")][0] * discs_of[("4CH", "ES")][0]) * step / 1000
print("By hand:", "EDV", round(edv), "mL · ESV", round(esv), "mL · EF", round(camus.ejection_fraction(round(edv), round(esv)), 1), "%")
print("Course tool:", camus.measure("patient0201"))

**Python note.** `camus.discs` follows the reference script of CAMUS: The annulus is the longest nearly straight stretch of the outline, the apex the point farthest from its middle, and the 20 diameters are measured perpendicular to the line between them. `discs_of[("2CH", "ED")][0]` takes the first of the four returned values, the diameters. `max(x[1] for x in discs_of.values())` takes the largest thickness of the four outlines. The volumes are rounded to whole mL before the EF, as in the reference.

In [ ]:
# The long axis and the 20 discs on the four-chamber view at end-diastole
im, lab, sp = camus.load("patient0201", "4CH", "ED")
d, step_4ed, base, apex = discs_of[("4CH", "ED")]
axis = apex - base
fig, ax = plt.subplots(figsize=(4.4, 4.4))
ax.imshow(im, cmap="gray")
ax.contour(lab == 1, levels=[0.5], colors="#E5797B", linewidths=1)
ax.plot([base[1], apex[1]], [base[0], apex[0]], color="#F9AB00", lw=1.5)
normal = np.array([-axis[1], axis[0]]) / np.linalg.norm(axis)
for f, width in zip(np.linspace(0, 1, 20, endpoint=False), d):
    c = base + f * axis
    half = width / 2 / sp[0]
    ax.plot([c[1] - half * normal[1], c[1] + half * normal[1]], [c[0] - half * normal[0], c[0] + half * normal[0]], color="#7DB4DD", lw=0.8)
ax.set_title("4CH ED: long axis and discs (centred on the axis)", fontsize=8)
ax.axis("off")
plt.show()

In [ ]:
net_case = camus.measure("patient0201", net_masks["patient0201"])
print(pd.DataFrame([camus.measure("patient0201"), net_case], index=["expert", "network"]))

**Observe and reflect.** The network draws a larger end-diastolic and a larger end-systolic volume than the expert, and its EF is still close. Which clinical decision depends on the volumes and not only on the EF?

**Exercise H.** Measure patient 201 on the expert outlines with a thickness for each phase, `camus.measure("patient0201", thickness="phase")`. What is the end-systolic volume in mL?

In [ ]:
data = exercise_table("H")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = camus.measure("patient0201", thickness="phase")["esv_ml"]
teach.check("Exercise H", answer)

## 9. Agreement with the expert: Bland-Altman

*Lecture: [Agreement with the expert: Bland-Altman](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#agreement-with-the-expert-bland-altman)*

Two methods of measurement can correlate closely and still disagree by many points. Bland and Altman plot the difference of the two methods against their mean. The mean difference, the bias, shows a systematic error; the limits of agreement, the bias plus and minus 1.96 standard deviations of the differences, show how far one measurement of a patient may fall from the other. A laboratory decides before the test which limits it accepts.

**From the formulas to the code.** With $d_i = EF_{net,i} - EF_{exp,i}$ for the 50 test patients: $\bar{d}$ is `diff.mean()`, $s_d$ is `diff.std(ddof=1)` and the limits are $\bar{d} \pm 1.96\,s_d$. Pearson's $r$ is `np.corrcoef(ef_net, ef_expert)[0, 1]`. `ef_range` sorts each EF into the three ranges of the guideline with `np.select`, which takes the first condition that holds: 40 percent or less, above 40 and below 50 percent, and 50 percent or more.

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(9)

In [ ]:
rows = []
for p in camus.patients("testing"):
    ref, net = camus.measure(p), camus.measure(p, net_masks[p])
    rows.append(dict(patient=p, ef_expert=ref["ef"], ef_net=net["ef"], edv_expert=ref["edv_ml"], edv_net=net["edv_ml"],
                     esv_expert=ref["esv_ml"], esv_net=net["esv_ml"]))
agreement = pd.DataFrame(rows)
diff = agreement["ef_net"] - agreement["ef_expert"]
bias, sd = float(diff.mean()), float(diff.std(ddof=1))
r = float(np.corrcoef(agreement["ef_net"], agreement["ef_expert"])[0, 1])
print("EF: bias", round(bias, 2), "· SD", round(sd, 2), "· limits", round(bias - 1.96 * sd, 1), "to", round(bias + 1.96 * sd, 1), "· r", round(r, 3))

**Python note.** Measuring 100 sets of four outlines, 50 patients by the expert and by the network, takes about a minute, because the method of discs searches the annulus on every outline. `ddof=1` divides by n − 1, as for a sample. The table `agreement` keeps the volumes too, so that the agreement of the volumes can be studied in the weekly project.

In [ ]:
mean_ef = (agreement["ef_net"] + agreement["ef_expert"]) / 2
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.8))
axes[0].scatter(agreement["ef_expert"], agreement["ef_net"], s=16, color="#1F5F8B")
axes[0].plot([10, 80], [10, 80], color="#888888", lw=1)
axes[0].set_xlabel("EF, expert (%)")
axes[0].set_ylabel("EF, network (%)")
axes[0].set_title(f"r = {r:.2f}", fontsize=9)
axes[1].scatter(mean_ef, diff, s=16, color="#1F5F8B")
for value, style in ((bias, "-"), (bias - 1.96 * sd, "--"), (bias + 1.96 * sd, "--")):
    axes[1].axhline(value, color="#9B1B1B", ls=style, lw=1)
axes[1].set_xlabel("mean of the two EFs (%)")
axes[1].set_ylabel("network minus expert (points)")
axes[1].set_title("Bland-Altman", fontsize=9)
plt.tight_layout()
plt.show()
labels = ["≤40", "41-49", "≥50"]


def ef_range(ef):
    """The range of each EF in the heart failure guideline: 40 or less, above 40 and below 50, 50 or more."""
    return pd.Categorical(np.select([ef <= 40, ef < 50], labels[:2], labels[2]), categories=labels)


print(pd.crosstab(ef_range(agreement["ef_expert"]), ef_range(agreement["ef_net"]),
                  rownames=["expert"], colnames=["network"], dropna=False))

**Observe and reflect.** A correlation of about 0.8 sounds good, yet the limits of agreement span about 30 points, and about a third of the patients change their range of EF. Which patients of the cross table matter most for a decision on therapy, and what would you require before the automatic EF enters a report?

**Exercise I.** Measure the 50 test patients on the expert outlines twice, with `thickness="phase"` and with `thickness="camus"`. What is the mean difference of the EF, phase minus camus, in percentage points? Round to two decimals.

In [ ]:
data = exercise_table("I")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
phase_minus_camus = [camus.measure(p, thickness="phase")["ef"] - camus.measure(p, thickness="camus")["ef"]
                     for p in camus.patients("testing")]
answer = round(float(np.mean(phase_minus_camus)), 2)
teach.check("Exercise I", answer)

## 10. Working with an AI assistant, part 4: Reviewing a measurement pipeline

*Lecture: [Working with an AI assistant, part 4: Reviewing a measurement pipeline](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lecture.html#ai-pair-programming-part-4-reviewing-a-measurement-pipeline)*

A participant asked an AI assistant to *measure the EF of patient 201 from the outlines and to report the agreement of the network with the expert*. The code below is its answer. It runs and prints an EF that looks plausible. Before trusting it, read it with the six questions of the review of a measurement pipeline: Are the sets split by patient? How were the masks resized? Is the real size of a pixel used? Is the Dice coefficient given by view and phase? Which rule gives the volumes? Is agreement reported, or only correlation?

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB04_transfer_learning.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(10)

In [ ]:
# Code of an AI assistant: the EF of patient 201, measured on the frames of 96 pixels
small = camus.frames(["patient0201"], size=96)
outlines = {(v, ph): small["masks"][i] > 0 for i, (v, ph) in enumerate(zip(small["view"], small["phase"]))}
pixel = (1.0, 1.0)
edv_a, esv_a = camus.volumes(outlines[("2CH", "ED")], outlines[("2CH", "ES")], pixel,
                             outlines[("4CH", "ED")], outlines[("4CH", "ES")], pixel)
print("EF:", round(camus.ejection_fraction(edv_a, esv_a), 1), "%")
# and for the 50 test patients the assistant reported:
print("Agreement with the expert: r =", round(float(np.corrcoef(agreement["ef_net"], agreement["ef_expert"])[0, 1]), 2))

The code has three problems and one omission. `> 0` keeps every label, so the outline holds the myocardium and the left atrium together with the cavity. The masks of 96 pixels are measured as if a pixel were 1 mm, while a pixel of the resized frame is about 1.7 by 2.0 mm; the EF hides this error because it is a ratio, but every volume is wrong. And a correlation is reported as agreement. The omission: The assistant did not print the volumes, which would have shown at once that something is wrong.

**Exercise J.** Correct the pipeline: Measure patient 201 on the expert cavity, label 1, at the original size of the frames and with the real size of a pixel. What is the end-diastolic volume in mL?

In [ ]:
data = exercise_table("J")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = camus.measure("patient0201")["edv_ml"]
teach.check("Exercise J", answer)

The [prompt workshop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lab.html#part-f) of the interactive lab writes the prompt of such a measurement with the seven fields of the prompt card. The field *Data* names the size of a pixel and the meaning of the labels, and the field *Test* asks for the volumes next to the EF.

## Application challenges by field

| Field | Challenge |
|---|---|
| Heart failure | Use the table `agreement` of Section 9 and count the patients whose range of EF (up to 40, 41 to 49, 50 or more percent) differs between the network and the expert. For which of these patients would the difference change a decision on therapy? |
| Interventional cardiology | CAMUS also outlines the left atrium, label 3. Train the U-Net of Section 7 on `masks == 3` and report the Dice coefficient of the left atrium at end-systole. Why does a structural heart team care about the size of the atrium? |
| Cardiovascular imaging | Train the U-Net with the first 100 instead of 200 training patients and compare the Dice coefficient by view and phase. How many outlined examinations would a new laboratory need? |
| Echocardiography laboratory | Compare the error of the EF of patients whose two views are of good quality with that of the others, using `camus_table`. Should a laboratory exclude poor windows before an automatic measurement, and what would it lose? |
| Cardio-oncology | During cancer therapy, a follow-up echocardiogram shows an EF 10 percentage points lower than before. Use the standard deviation of the differences of Section 9: How large could the difference between two automatic measurements of an unchanged heart be, if each differs from the expert independently? |
| Electrophysiology | In atrial fibrillation the cycle length changes from beat to beat, and the guidelines recommend measuring over several beats. CAMUS gives one end-diastolic and one end-systolic frame per view. What would you change in the pipeline before using it in patients with atrial fibrillation? |
| Intensive care | Time the U-Net on the four frames of one patient on your computer, including `camus.to_original` and `camus.measure`. Could an automatic EF be ready at the bedside before the sonographer leaves the room? |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-04/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of Week 4, described on the [week page](README.md), measures the EF of the 50 test patients of CAMUS with the U-Net of Section 7 and judges it as an echocardiography laboratory would.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>